# Scorecard de Produtos Perenes — INSIDER

Notebook unificado para avaliação contínua do portfólio de produtos perenes.

**Arquitetura idêntica ao Scorecard de Lançamentos** — mesmo motor de scoring, mesmas funções utilitárias.

### Seções
0. **Setup e Configurações**
1. **Gathering Data** — queries BQ, consolidação da base
2. **Compute Scorecard** — motor JSON-driven com scores por pilar
3. **Checklist Operacional** — avaliação binária (passa / não passa)
4. **Análises de Negócio** — visualizações reportáveis
5. **Exportação** — Google Sheets

In [1]:
!pip install gspread==6.2.1


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from datetime import date
import pandas as pd
import numpy as np
import gspread
from google.oauth2.service_account import Credentials

import os
import json
from pathlib import Path
import re

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

service_account_info = os.getenv("BQ_SERVICE_ACCOUNT")

# Define the scope for the Google Sheets API
scope = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive',
]

# Authentication - using a service account file
def get_service_account_credentials():
    import os
    import json

    # Path to the service account file
    return json.loads(os.getenv("BIGQUERY_INTEGRATION_SERVICE_ACCOUNT"))

# Extracting credentials
credentials_info = get_service_account_credentials()
cred_bundle = Credentials.from_service_account_info(credentials_info, scopes=scope)

# Authorize the client
client = gspread.authorize(cred_bundle)

# Open the Google Spreadsheet
SPREADSHEET_SCORECARD = client.open_by_url('https://docs.google.com/spreadsheets/d/1PjR2Czi9pYCMoMXPaI7QCHJIUmlN-7AeyXLAC91R7GQ/')

In [3]:
# ── Garantir working directory = pasta do notebook ─────────────────
_notebook_dir = Path(globals().get("__vsc_ipynb_file__", os.path.abspath("scorecard_perenes.ipynb"))).parent
os.chdir(_notebook_dir)
print(f"Working dir: {_notebook_dir}")


# ── Inputs manuais do scorecard ───────────────────────────────────
SCORECARD_INPUTS = {
    "benchmark_mc3": 0.20,
    "benchmark_mrkup_target": 3.85,
}

# ── Critérios do checklist (avaliação binária) ────────────────────
CHECKLIST_CRITERIA = {
    "Unit Economics": {
        "MC3 mínimo": {"column": "metric_mc3", "check_value": 0.15, "comparison": "greater_equal"},
        "Markup target": {"column": "metric_markup_vs_target", "check_value": 0.80, "comparison": "greater_equal"},
    },
    "Saúde de Estoque": {
        "Cobertura < 180d": {"column": "metric_cobertura_dias", "check_value": 180, "comparison": "less_equal"},
        "CV vendas SKC < 1.2": {"column": "metric_cv_vendas_skc", "check_value": 1.20, "comparison": "less_equal"},
        "Disponibilidade > 70%": {"column": "metric_disponibilidade_sku_semana", "check_value": 0.70, "comparison": "greater_equal"},
    },
    "Satisfação do Cliente": {
        "T&D ≤ 1.5x categoria": {"column": "metric_t_e_d_vs_categoria", "check_value": 1.50, "comparison": "less_equal"},
    },
}

# ── Path do JSON de configuração ──────────────────────────────────
SCORECARD_CONFIG_PATH = Path("inputs/JSON/params.json")

# ── Paleta de cores para classificações ───────────────────────────
COLOR_MAP_CLASSIFICACAO = {
    "INVEST": "#2ecc71",
    "KEEP":   "#3498db",
    "WATCH":  "#f39c12",
    "AT RISK":   "#e74c3c",
}

# ── Routing de acionáveis por pilar ───────────────────────────────
PILAR_RESPONSAVEL = {
    "unit_economics":      "Revenue Management",
    "estoque":             "IOP/S&OE",
    "tracao_comercial":    "Growth/Mídia",
    "satisfacao_e_marca":  "Produto Físico",
}

Working dir: /datasets/_deepnote_work


## 1. Carregar Base de Perenes

Lê os parâmetros do `params.json`, injeta nas queries SQL, executa contra o BigQuery
e consolida tudo em `df_base`.

In [4]:
with open(SCORECARD_CONFIG_PATH) as f:
    scorecard_params = json.load(f)

params = scorecard_params

# ── Injetar parâmetros nas queries SQL ─────────────────────────────
lookback_months=params["time_windows"]["base_months"]
full_price_max_disc=params["full_price_definition"]["max_discount_pct"]
tendencia_window_months=params["time_windows"]["tendencia_window_months"]
skc_low_giro_days=params["time_windows"]["skc_low_giro_days"]
ruptura_semanas=params["time_windows"]["ruptura_semanas"]
cobertura_fallback_days=params["time_windows"]["cobertura_fallback_days"]
recompra_days=params["time_windows"]["recompra_days"]
ltv_days=params["time_windows"]["ltv_days"]
lookback_months=params["time_windows"]["base_months"]
markup_target=params["markup_target"]

# ── Executar queries ──────────────────────────────────────────────


In [5]:
def read_sql_file(file_path: str) -> str:
    with open(file_path, "r") as f:
        return f.read()
        
sql_dre = read_sql_file("inputs/SQL/01_base_dre.sql").format(
    lookback_months=params["time_windows"]["base_months"],
    full_price_max_disc=params["full_price_definition"]["max_discount_pct"],
    tendencia_window_months=params["time_windows"]["tendencia_window_months"],
)

sql_estoque = read_sql_file("inputs/SQL/02_estoque.sql").format(
    skc_low_giro_days=params["time_windows"]["skc_low_giro_days"],
    ruptura_semanas=params["time_windows"]["ruptura_semanas"],
    cobertura_fallback_days=params["time_windows"]["cobertura_fallback_days"],
)

sql_recompra = read_sql_file("inputs/SQL/03_recompra_ltv.sql").format(
    recompra_days=params["time_windows"]["recompra_days"],
    ltv_days=params["time_windows"]["ltv_days"],
)

sql_markup = read_sql_file("inputs/SQL/04_markup.sql").format(
    lookback_months=params["time_windows"]["base_months"],
    markup_target=params["markup_target"],
)

In [6]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_dre = _dntk.execute_sql(
  '-- =============================================================================\n-- 01_base_dre.sql\n-- Gathering de dados da DRE para Unit Economics e Tração Comercial\n-- Granularidade: product_name x mes\n-- =============================================================================\n-- Parâmetros injetados via Python (f-string ou .format()):\n--   6           → janela base (default: 6)\n--   0.2       → threshold de full price (default: 0.20)\n--   5   → janela de trend com trimming (default: 5)\n-- =============================================================================\n\nWITH\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- BASE: pedidos válidos de perenes, consolidando backorders\n-- ─────────────────────────────────────────────────────────────────────────────\nbase AS (\n  SELECT\n    COALESCE(dre.parent_order_id, dre.order_id)            AS order_id,\n    IF(SUM(CAST(dre.is_child_order AS INT64)) > 0, TRUE, FALSE) AS has_backorder,\n    dre.insider_customer_id,\n    dre.order_number,\n    DATE_TRUNC(CAST(dre.order_date AS DATE), MONTH)        AS order_month,\n    s.product_name,\n    dre.category_1,\n    dre.category_2,\n    dre.category_3,\n    dre.category_4,\n\n    -- Receita e margem\n    SUM(dre.items_potential_revenue)                        AS items_potential_revenue,\n    SUM(dre.revenue_after_discounts)                        AS revenue_after_discounts,\n    SUM(dre.revenue_after_refunds)                          AS revenue_after_refunds,\n    sum(net_profit_after_marketing_costs)                   AS net_profit_after_marketing_costs,\n    sum(nullif(revenue_after_taxes, 0))                     AS revenue_after_taxes, \n\n    SUM(dre.non_refunded_quantity)                          AS non_refunded_quantity,\n    SUM(dre.sold_quantity)                                  AS sold_quantity,\n    SUM(dre.refunded_quantity)                              AS refunded_quantity,\n    SUM(dre.exchanged_quantity)                             AS exchanged_quantity\n\n  FROM `insider-data-lake.fpa.analytical_dre` AS dre\n  LEFT JOIN `insider-data-lake.integrated.skus` AS s\n    USING (sku)\n\n  WHERE TRUE\n    AND dre.order_status != \'Not authorized\'\n    AND LOWER(s.product_name) NOT LIKE \'%kit%\'\n    AND LOWER(s.product_name) NOT LIKE \'%defeito%\'\n    AND s.sku_state NOT IN (\'personalizacao\', \'ativo_personalizacao\')\n    AND s.sku_state = \'ativo_perene\'                       -- apenas perenes\n    AND dre.order_date between DATETIME_SUB(DATETIME_TRUNC(CURRENT_DATETIME(), MONTH), INTERVAL 6 MONTH)\n                        AND DATETIME_TRUNC(CURRENT_DATETIME(), MONTH)\n\n  GROUP BY ALL\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- MÉTRICAS POR PRODUTO (agregado total do período)\n-- ─────────────────────────────────────────────────────────────────────────────\nproduct_totals AS (\n  SELECT\n    product_name,\n    MAX(category_1)                                         AS category_1,\n    MAX(category_2)                                         AS category_2,\n    MAX(category_3)                                         AS category_3,\n    MAX(category_4)                                         AS category_4,\n\n    -- Unit Economics\n    safe_divide(\n      sum(net_profit_after_marketing_costs), \n      sum(nullif(revenue_after_taxes, 0))\n    )                                                       AS metric_mc3,\n\n    SUM(revenue_after_discounts)                              AS receita_total,\n    safe_divide(\n      sum(net_profit_after_marketing_costs),\n      sum(non_refunded_quantity)\n    )                                                       AS contribuicao_absoluta,\n    SUM(non_refunded_quantity)                              AS unidades_total,\n\n    -- Full price (desconto <= 0.2)\n    SAFE_DIVIDE(\n      SUM(IF(\n        SAFE_DIVIDE(revenue_after_discounts, \n        NULLIF(items_potential_revenue, 0)) >= (1 - 0.2),\n        non_refunded_quantity, 0\n      )),\n      SUM(non_refunded_quantity)\n    )                                                       AS metric_full_price_pct,\n\n    -- T&D\n    SAFE_DIVIDE(\n      SUM(refunded_quantity + exchanged_quantity),\n      NULLIF(SUM(sold_quantity), 0)\n    )                                                       AS t_e_d_produto\n\n  FROM base\n  GROUP BY product_name\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- T&D NORMALIZADO PELA CATEGORIA\n-- ─────────────────────────────────────────────────────────────────────────────\ncategory_trd AS (\n  SELECT\n    category_4,\n    SAFE_DIVIDE(\n      SUM(refunded_quantity + exchanged_quantity),\n      NULLIF(SUM(sold_quantity), 0)\n    ) AS t_e_d_categoria\n  FROM base\n  GROUP BY category_4\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- RECEITA MENSAL POR PRODUTO (para tendência MoM e percentil)\n-- ─────────────────────────────────────────────────────────────────────────────\nproduct_monthly AS (\n  SELECT\n    product_name,\n    order_month,\n    SUM(revenue_after_discounts) AS receita_mensal\n  FROM base\n  GROUP BY product_name, order_month\n),\n\ntb_receita_media_mensal AS (\n  SELECT\n    product_name,\n    AVG(receita_mensal)     AS receita_media_mensal,\n    COUNT(order_month)      AS n_meses_com_venda\n  FROM product_monthly\n  GROUP BY product_name\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- SHARE-OF-PORTFOLIO TREND (substitui tendência MoM absoluta e percentil)\n-- Calcula share do produto na receita total do portfólio perene por mês,\n-- depois tendência MoM do share com trimming de outliers (remove mês max e min)\n-- ─────────────────────────────────────────────────────────────────────────────\nreceita_total_mensal AS (\n  SELECT\n    order_month,\n    SUM(revenue_after_discounts) AS receita_portfolio_mes\n  FROM base\n  GROUP BY order_month\n),\n\nproduct_share_mensal AS (\n  SELECT\n    pm.product_name,\n    pm.order_month,\n    pm.receita_mensal,\n    rtm.receita_portfolio_mes,\n    SAFE_DIVIDE(pm.receita_mensal, NULLIF(rtm.receita_portfolio_mes, 0)) AS share_no_portfolio\n  FROM product_monthly pm\n  JOIN receita_total_mensal rtm USING (order_month)\n  WHERE pm.order_month < DATE_TRUNC(CURRENT_DATE(), MONTH)\n),\n\nshare_with_prev AS (\n  SELECT\n    product_name,\n    order_month,\n    share_no_portfolio,\n    LAG(share_no_portfolio) OVER (PARTITION BY product_name ORDER BY order_month) AS share_prev\n  FROM product_share_mensal\n  WHERE order_month >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 4 MONTH)\n),\n\nshare_trend AS (\n  SELECT\n    product_name,\n    SUM(\n      CASE\n        WHEN SAFE_DIVIDE(share_no_portfolio, NULLIF(share_prev, 0)) < 0.9  THEN -1\n        WHEN SAFE_DIVIDE(share_no_portfolio, NULLIF(share_prev, 0)) < 1.1  THEN  0\n        WHEN SAFE_DIVIDE(share_no_portfolio, NULLIF(share_prev, 0)) > 1.1  THEN  1\n        ELSE NULL\n      END\n    )                 AS metric_share_trend,\n    COUNT(share_prev) AS n_meses_trend\n  FROM share_with_prev\n  WHERE share_prev IS NOT NULL\n  GROUP BY product_name\n),\n\n-- Percentil de receita movido para health metrics (informativo, sem peso no score)\npercentis AS (\n  SELECT\n    product_name,\n    PERCENT_RANK() OVER (ORDER BY receita_media_mensal ASC) AS health_percentil_receita\n  FROM tb_receita_media_mensal\n)\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- OUTPUT FINAL\n-- ─────────────────────────────────────────────────────────────────────────────\nSELECT\n  pt.product_name,\n  pt.category_1,\n  pt.category_2,\n  pt.category_3,\n  pt.category_4,\n  pt.receita_total,\n  pt.unidades_total,\n\n  -- Unit Economics\n  pt.metric_mc3,\n  pt.contribuicao_absoluta,\n  pt.metric_full_price_pct,\n\n  -- Tração\n  rmm.receita_media_mensal,\n  p.health_percentil_receita,\n  st.metric_share_trend,\n\n  -- Satisfação (T&D)\n  pt.t_e_d_produto,\n  ct.t_e_d_categoria,\n  SAFE_DIVIDE(pt.t_e_d_produto, NULLIF(ct.t_e_d_categoria, 0)) AS metric_t_e_d_vs_categoria\n\nFROM product_totals pt\nLEFT JOIN tb_receita_media_mensal rmm USING (product_name)\nLEFT JOIN percentis p             USING (product_name)\nLEFT JOIN share_trend st          USING (product_name)\nLEFT JOIN category_trd ct\n  ON pt.category_4 = ct.category_4\n\nORDER BY pt.receita_total DESC',
  'SQL_E1199E86_8709_489E_A277_2784CCE6BB51',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_dre

,product_name,category_1,category_2,category_3,category_4,receita_total,unidades_total,metric_mc3,contribuicao_absoluta,metric_full_price_pct,receita_media_mensal,health_percentil_receita,metric_share_trend,t_e_d_produto,t_e_d_categoria,metric_t_e_d_vs_categoria
0,Tech T-shirt Gola U Masculino,Man,Man Casual,Man Casual Top,Man Casual Top T-Shirt,2.437765e+07,208586.420567,0.209256,20.461175,0.278084,4.062942e+06,1.000000,0.0,0.060253,0.067155,0.897223
1,Daily T-shirt Masculino,Man,Man Casual,Man Casual Top,Man Casual Top T-Shirt,1.884322e+07,236448.262361,0.245851,16.232176,0.079197,3.140537e+06,0.989130,1.0,0.057180,0.067155,0.851465
2,The Perfect Top Feminino,Woman,Woman Casual,Woman Casual Top,Woman Casual Top Tank Top,1.658082e+07,157226.651148,0.029221,2.546537,0.402459,2.763471e+06,0.978261,0.0,0.075069,0.080953,0.927322
3,Wingsuit Feminino,Woman,Woman Casual,Woman Casual Top,Woman Casual Top Jackets/Hoodies,1.015117e+07,26754.720163,0.424268,136.002011,0.635326,1.691862e+06,0.967391,1.0,0.070786,0.070786,1.000000
4,Daily Light T-shirt Masculino,Man,Man Casual,Man Casual Top,Man Casual Top T-Shirt,1.005036e+07,140865.119215,0.244171,14.548545,0.999990,1.675061e+06,0.956522,-3.0,0.077431,0.067155,1.153017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,Regata Nadador IN-ACTION Seamless Feminino,Woman,Woman Fitness/Sportswear,Woman Fitness/Sportswear Top,Woman Fitness/Sportswear Top Tank Top,1.005112e+05,673.675192,0.240415,30.369167,0.734394,2.010224e+04,0.065217,-3.0,0.144660,0.122452,1.181360
89,Undershirt Anti Suor Gola U Masculino,Man,Man Underwear,Man Underwear Top,Man Underwear Top Undershirt,5.840200e+04,440.601656,0.250952,27.851205,0.665021,1.168040e+04,0.032609,-1.0,0.048692,0.037547,1.296823
90,Undershirt Simples Gola V Masculino,Man,Man Underwear,Man Underwear Top,Man Underwear Top Undershirt,2.991029e+04,239.000000,0.202667,20.924844,0.878661,5.982058e+03,0.010870,1.0,0.032389,0.037547,0.862612
91,Pochete Slim Esportiva SprintIn,Unissex,Unissex Accessory,Unissex Accessory Bag,Unissex Accessory Bag Pochete,2.214459e+04,183.000000,0.114655,11.573176,0.726776,7.381528e+03,0.021739,NaN,0.016216,0.016216,1.000000


In [7]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_estoque = _dntk.execute_sql(
  '-- =============================================================================\n-- 02_estoque.sql\n-- Cobertura de estoque, desequilíbrio entre SKUs e disponibilidade por produto\n-- Granularidade de output: product_name\n-- =============================================================================\n-- Fontes:\n--   - sop_gold.stock_health (estoque, cobertura em dias, vendas L7D)\n--   - insider-data-lake.fpa.analytical_dre (vendas por SKU)\n--   - integrated.skus (filtro ativo_perene)\n--\n-- Parâmetros injetados via Python:\n--   60       → janela para CV de vendas entre SKUs (default: 60)\n--   26         → janela para % semanas com ruptura (default: 26)\n--   90 → janela da média móvel de vendas como fallback (default: 90)\n-- =============================================================================\n\nWITH\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- ESTOQUE ATUAL POR SKU (via stock_health no dia = CURRENT_DATE)\n-- ─────────────────────────────────────────────────────────────────────────────\nstock_today AS (\n  SELECT\n    sh.produto_pai                     AS product_name,\n    sh.sku,\n    sh.estoque_passado_ou_projetado    AS estoque_unidades,\n    sh.estoque_passado_ou_projetado_d  AS estoque_dias,\n    sh.qtd_venda_media_l7d,\n    sh.receita_prevista_diarizada\n  FROM `insider-data-lake.sop_gold.stock_health` AS sh\n  JOIN `insider-data-lake.integrated.skus` AS s USING (sku)\n  WHERE sh.dia = CURRENT_DATE()\n    AND s.sku_state = \'ativo_perene\'\n    AND LOWER(sh.produto_pai) NOT LIKE \'%kit%\'\n    AND LOWER(sh.produto_pai) NOT LIKE \'%defeito%\'\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- VENDAS POR SKU NOS ÚLTIMOS N DIAS (para CV de vendas entre SKUs)\n-- ─────────────────────────────────────────────────────────────────────────────\nvendas_por_sku AS (\n  SELECT\n    s.product_name,\n    s.color,\n    SUM(dre.non_refunded_quantity) AS total_vendido\n  FROM `insider-data-lake.fpa.analytical_dre` AS dre\n  JOIN `insider-data-lake.integrated.skus` AS s USING (sku)\n  WHERE dre.order_status != \'Not authorized\'\n    AND dre.order_date >= DATETIME_SUB(CURRENT_DATETIME(), INTERVAL 60 DAY)\n    AND s.sku_state = \'ativo_perene\'\n    AND LOWER(s.product_name) NOT LIKE \'%kit%\'\n    AND LOWER(s.product_name) NOT LIKE \'%defeito%\'\n  GROUP BY s.product_name, s.color\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- COEFICIENTE DE VARIAÇÃO DE VENDAS ENTRE SKUs (menor = mais equilibrado)\n-- Substitui a métrica de % SKCs com baixo giro (mais granular e informativa)\n-- ─────────────────────────────────────────────────────────────────────────────\ncv_por_produto AS (\n  SELECT\n    product_name,\n    SAFE_DIVIDE(\n      STDDEV(total_vendido),\n      NULLIF(AVG(total_vendido), 0)\n    ) AS metric_cv_vendas_skc,\n    COUNT(*) AS n_skcs_com_venda,\n    SUM(total_vendido) AS vendas_total_periodo\n  FROM vendas_por_sku\n  GROUP BY product_name\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- MÉTRICAS DE GRADE POR PRODUTO (contagem de SKUs ativos e estoque total)\n-- ─────────────────────────────────────────────────────────────────────────────\ngrade_metrics AS (\n  SELECT\n    product_name,\n    SUM(estoque_unidades)             AS estoque_total_unidades,\n  FROM stock_today\n  GROUP BY product_name\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- FALLBACK: VENDA DIÁRIA MÉDIA ÚLTIMOS 90 DIAS (quando forecast indisponível)\n-- ─────────────────────────────────────────────────────────────────────────────\nvenda_media_fallback AS (\n  SELECT\n    s.product_name,\n    SAFE_DIVIDE(\n      SUM(dre.non_refunded_quantity),\n      90\n    ) AS venda_diaria_fallback\n  FROM `insider-data-lake.fpa.analytical_dre` AS dre\n  JOIN `insider-data-lake.integrated.skus` AS s USING (sku)\n  WHERE dre.order_status != \'Not authorized\'\n    AND dre.order_date >= DATETIME_SUB(CURRENT_DATETIME(), INTERVAL 90 DAY)\n    AND s.sku_state = \'ativo_perene\'\n    AND LOWER(s.product_name) NOT LIKE \'%kit%\'\n    AND LOWER(s.product_name) NOT LIKE \'%defeito%\'\n  GROUP BY s.product_name\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- COBERTURA DE ESTOQUE (usa forecast como denominador; fallback = MA 90d)\n-- ─────────────────────────────────────────────────────────────────────────────\ncobertura AS (\n  SELECT\n    st.product_name,\n    SUM(st.qtd_venda_media_l7d) AS venda_diaria_media,\n    vmf.venda_diaria_fallback,\n    CASE\n      WHEN COALESCE(SUM(st.receita_prevista_diarizada), 0) = 0\n           AND COALESCE(vmf.venda_diaria_fallback, 0) = 0\n           AND SUM(st.estoque_unidades) > 0\n        THEN 9999\n      WHEN COALESCE(SUM(st.estoque_unidades), 0) = 0\n        THEN 0\n      WHEN SUM(st.receita_prevista_diarizada) > 0\n        THEN SAFE_DIVIDE(\n          SUM(st.estoque_dias * st.receita_prevista_diarizada),\n          NULLIF(SUM(st.receita_prevista_diarizada), 0)\n        )\n      ELSE SAFE_DIVIDE(\n        SUM(st.estoque_unidades),\n        vmf.venda_diaria_fallback\n      )\n    END AS metric_cobertura_dias,\n    CASE\n      WHEN SUM(st.receita_prevista_diarizada) > 0 THEN \'forecast\'\n      WHEN vmf.venda_diaria_fallback > 0 THEN \'media_90d\'\n      ELSE \'sem_denominador\'\n    END AS cobertura_fonte\n  FROM stock_today st\n  LEFT JOIN venda_media_fallback vmf ON st.product_name = vmf.product_name\n  GROUP BY st.product_name, vmf.venda_diaria_fallback\n),\n-- ─────────────────────────────────────────────────────────────────────────────\n-- TAMANHO MAIS VENDIDO\n-- Captura o tamanho mais vendido por produto, para filtrar SKUs de interesse\n-- ─────────────────────────────────────────────────────────────────────────────\nvenda_por_tamanho AS (\n  SELECT\n    sh.produto_pai                     AS product_name,\n    s.size,\n    sum(sh.qtd_venda_media_l7d)        AS venda_media_l7d\n  FROM `insider-data-lake.sop_gold.stock_health` AS sh\n  JOIN `insider-data-lake.integrated.skus` AS s USING (sku)\n  WHERE sh.dia >= DATE_SUB(CURRENT_DATE(), INTERVAL 26 WEEK)\n    AND s.sku_state = \'ativo_perene\'\n  GROUP BY sh.produto_pai, s.size\n),\n\ntamanho_mais_vendido AS (\n  SELECT\n    product_name,\n    size,\n    venda_media_l7d\n  FROM venda_por_tamanho\n  QUALIFY ROW_NUMBER() OVER (\n      PARTITION BY product_name\n      ORDER BY venda_media_l7d DESC\n    ) = 1\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- RUPTURA SKU × DIA\n-- (nº de combinações SKU×dia com estoque = 0) / (total combinações possíveis)\n-- Captura disponibilidade a nível de grade completa, não apenas produto agregado\n-- ─────────────────────────────────────────────────────────────────────────────\nestoque_semanal AS (\n  SELECT\n    sh.produto_pai                     AS product_name,\n    DATE_TRUNC(sh.dia, WEEK)           AS semana,\n    SUM(\n      CASE\n        WHEN sh.estoque_passado_ou_projetado > 0 THEN 1\n        ELSE 0\n      END\n    ) AS skus_com_estoque_na_semana,\n    COUNT(1) AS total_skus_na_semana\n  FROM `insider-data-lake.sop_gold.stock_health` AS sh\n  JOIN `insider-data-lake.integrated.skus` AS s USING (sku)\n  INNER JOIN tamanho_mais_vendido AS tmv\n    ON sh.produto_pai = tmv.product_name\n    AND s.size = tmv.size\n  WHERE sh.dia >= DATE_SUB(CURRENT_DATE(), INTERVAL 26 WEEK)\n    AND s.sku_state = \'ativo_perene\'\n\n  GROUP BY sh.produto_pai, semana\n),\n\ndisponibilidade AS (\n  SELECT\n    product_name,\n    COUNT(DISTINCT semana) AS n_semanas_total,\n    SUM(skus_com_estoque_na_semana) AS skus_com_estoque_na_semana_total,\n    SUM(total_skus_na_semana) AS total_skus_na_semana_total,\n    \n    SAFE_DIVIDE(\n      SUM(skus_com_estoque_na_semana),\n      COUNT(DISTINCT semana)\n    ) AS skus_com_estoque_na_semana_avg,\n    \n    SAFE_DIVIDE(\n      SUM(total_skus_na_semana),\n      COUNT(DISTINCT semana)\n    ) AS total_skus_na_semana_avg,\n    \n    SAFE_DIVIDE(\n      SUM(skus_com_estoque_na_semana),\n      SUM(total_skus_na_semana)\n    ) AS metric_disponibilidade_sku_semana\n  FROM estoque_semanal\n  GROUP BY product_name\n)\n\n, numero_cores_por_produto as (\n  select \n    product_name,\n    count(distinct color) as n_skcs_total\n  from `insider-data-lake.integrated.skus`\n\n  group by product_name\n)\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- OUTPUT FINAL\n-- ─────────────────────────────────────────────────────────────────────────────\nSELECT\n  c.product_name,\n  gm.estoque_total_unidades,\n  c.venda_diaria_media,\n  c.metric_cobertura_dias,\n  c.cobertura_fonte,\n\n  cv.n_skcs_com_venda,\n  cv.metric_cv_vendas_skc,\n  ncp.n_skcs_total,\n\n  d.n_semanas_total,\n  d.metric_disponibilidade_sku_semana\n\nFROM cobertura c\nLEFT JOIN grade_metrics gm    USING (product_name)\nLEFT JOIN cv_por_produto cv   USING (product_name)\nLEFT JOIN disponibilidade d   USING (product_name)\nLEFT JOIN numero_cores_por_produto ncp   USING (product_name)\n\nORDER BY c.metric_cobertura_dias DESC',
  'SQL_E1199E86_8709_489E_A277_2784CCE6BB51',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_estoque

,product_name,estoque_total_unidades,venda_diaria_media,metric_cobertura_dias,cobertura_fonte,n_skcs_com_venda,metric_cv_vendas_skc,n_skcs_total,n_semanas_total,metric_disponibilidade_sku_semana


In [8]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_recompra = _dntk.execute_sql(
  '-- =============================================================================\n-- 03_recompra_ltv.sql\n-- Taxa de recompra, LTV temporal e #compras médias por produto\n-- Granularidade de output: product_name\n-- =============================================================================\n-- Parâmetros injetados via Python:\n--   180   → janela de recompra em dias (default: 365)\n--   720        → janela de LTV em dias (default: 365)\n-- =============================================================================\n\nWITH\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- BASE DE HISTÓRICO COMPLETO: todos os pedidos do cliente (SEM restrição de data)\n-- Necessário para calcular ordem_pedido corretamente, sem zerar clientes antigos\n-- ─────────────────────────────────────────────────────────────────────────────\nbase_historico_completo AS (\n  SELECT\n    dre.sku,\n    dre.insider_customer_id,\n    dre.order_id,\n    dre.parent_order_id,\n    dre.order_date,\n    dre.net_profit,\n    s.product_name\n  FROM `insider-data-lake.fpa.analytical_dre` dre\n  JOIN `insider-data-lake.integrated.skus` AS s USING (sku)\n  WHERE dre.order_status != \'Not authorized\'\n    AND dre.store = "shopify_insider-store-loja"\n    AND LOWER(s.product_name) NOT LIKE \'%kit%\'\n    AND LOWER(s.product_name) NOT LIKE \'%defeito%\'\n    AND s.sku_state NOT IN (\'personalizacao\', \'ativo_personalizacao\')\n    AND s.sku_state = \'ativo_perene\'\n    AND dre.order_date >= DATETIME_SUB(CURRENT_DATETIME(), INTERVAL 720 DAY)\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- BASE FILTRADA: apenas os últimos 720 dias (para LTV e recompra)\n-- ─────────────────────────────────────────────────────────────────────────────\nbase_fpa_dre AS (\n  SELECT *\n  FROM base_historico_completo\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- PEDIDOS CONSOLIDADOS POR CLIENTE (1 linha por pedido, sem inflação por item)\n-- Usa histórico COMPLETO para que ordem_pedido seja calculada corretamente\n-- ─────────────────────────────────────────────────────────────────────────────\npedidos_cliente AS (\n  SELECT\n    insider_customer_id,\n    COALESCE(parent_order_id, order_id)   AS unified_order_id,\n    MIN(order_date)                       AS order_ts,\n    SUM(net_profit)                       AS order_value\n  FROM base_historico_completo\n  GROUP BY 1, 2\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- ORDEM SEQUENCIAL DE CADA PEDIDO DO CLIENTE (para detectar 1ª compra geral)\n-- ─────────────────────────────────────────────────────────────────────────────\npedidos_numerados AS (\n  SELECT\n    *,\n    ROW_NUMBER() OVER (\n      PARTITION BY insider_customer_id\n      ORDER BY order_ts, unified_order_id\n    ) AS ordem_pedido\n  FROM pedidos_cliente\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- LTV TOTAL DO CLIENTE (para ltv_base referência)\n-- ─────────────────────────────────────────────────────────────────────────────\nltv_cliente AS (\n  SELECT\n    insider_customer_id,\n    SUM(order_value) AS ltv_valor\n  FROM pedidos_cliente\n  GROUP BY insider_customer_id\n),\n\nltv_base AS (\n  SELECT PERCENTILE_CONT(ltv_valor, 0.5) OVER () AS ltv_mediano_base\n  FROM ltv_cliente\n  LIMIT 1\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- PRESENÇA DE PRODUTO POR PEDIDO (pares únicos cliente + pedido + produto)\n-- ─────────────────────────────────────────────────────────────────────────────\nproduto_no_pedido AS (\n  SELECT DISTINCT\n    insider_customer_id,\n    COALESCE(parent_order_id, order_id)   AS unified_order_id,\n    product_name\n  FROM base_fpa_dre\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- PRIMEIRA COMPRA DE CADA CLIENTE POR PRODUTO (desempate determinístico)\n-- ─────────────────────────────────────────────────────────────────────────────\nprimeira_compra_produto AS (\n  SELECT\n    pp.insider_customer_id,\n    pp.product_name,\n    pp.unified_order_id   AS first_product_order_id,\n    pn.order_ts           AS first_product_order_ts,\n    pn.ordem_pedido       AS first_product_ordem\n  FROM produto_no_pedido pp\n  JOIN pedidos_numerados pn\n    ON pp.insider_customer_id = pn.insider_customer_id\n   AND pp.unified_order_id   = pn.unified_order_id\n  QUALIFY ROW_NUMBER() OVER (\n    PARTITION BY pp.insider_customer_id, pp.product_name\n    ORDER BY pn.order_ts, pn.unified_order_id\n  ) = 1\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- MÉTRICAS POR CLIENTE-PRODUTO (LTV e #compras nos dois recortes temporais)\n-- ─────────────────────────────────────────────────────────────────────────────\nmetricas_cliente_produto AS (\n  SELECT\n    fpp.insider_customer_id,\n    fpp.product_name,\n\n    -- LTV até a primeira compra do produto (incluindo o pedido do produto)\n    SUM(IF(\n      pn.order_ts < fpp.first_product_order_ts\n      OR (pn.order_ts = fpp.first_product_order_ts\n          AND pn.unified_order_id <= fpp.first_product_order_id),\n      pn.order_value, 0\n    )) AS ltv_ate_compra,\n\n    -- LTV antes da primeira compra do produto (excluindo o pedido do produto)\n    -- Forçado a 0 se a compra do produto foi o 1º pedido do cliente\n    IF(\n      fpp.first_product_ordem = 1,\n      0,\n      SUM(IF(\n        pn.order_ts < fpp.first_product_order_ts\n        OR (pn.order_ts = fpp.first_product_order_ts\n            AND pn.unified_order_id < fpp.first_product_order_id),\n        pn.order_value, 0\n      ))\n    ) AS ltv_pre_compra,\n\n    -- #compras até a primeira compra do produto (incluindo)\n    COUNTIF(\n      pn.order_ts < fpp.first_product_order_ts\n      OR (pn.order_ts = fpp.first_product_order_ts\n          AND pn.unified_order_id <= fpp.first_product_order_id)\n    ) AS compras_ate_compra,\n\n    -- #compras antes da primeira compra do produto (excluindo)\n    -- Forçado a 0 se a compra do produto foi o 1º pedido do cliente\n    IF(\n      fpp.first_product_ordem = 1,\n      0,\n      COUNTIF(\n        pn.order_ts < fpp.first_product_order_ts\n        OR (pn.order_ts = fpp.first_product_order_ts\n            AND pn.unified_order_id < fpp.first_product_order_id)\n      )\n    ) AS compras_pre_compra\n\n  FROM primeira_compra_produto fpp\n  JOIN pedidos_numerados pn\n    ON fpp.insider_customer_id = pn.insider_customer_id\n  GROUP BY 1, 2, fpp.first_product_order_ts, fpp.first_product_order_id,\n           fpp.first_product_ordem\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- PRIMEIRA COMPRA SIMPLES (para manter CTE de recompra inalterada)\n-- ─────────────────────────────────────────────────────────────────────────────\nprimeira_compra AS (\n  SELECT\n    insider_customer_id,\n    product_name,\n    first_product_order_ts AS primeira_compra_data\n  FROM primeira_compra_produto\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- FLAG DE RECOMPRA: cliente fez QUALQUER compra subsequente na Insider\n-- dentro da janela de 180 dias após comprar esse produto?\n-- (v2: mede se o produto gerou um recomprador da marca, não recompra do mesmo produto)\n-- ─────────────────────────────────────────────────────────────────────────────\nrecompra AS (\n  SELECT\n    pc.insider_customer_id,\n    pc.product_name,\n    MAX(IF(\n      ped.order_ts > pc.primeira_compra_data\n      AND ped.order_ts <= DATETIME_ADD(pc.primeira_compra_data, INTERVAL 180 DAY),\n      1, 0\n    )) AS recomprou\n  FROM primeira_compra pc\n  JOIN pedidos_cliente AS ped\n    ON pc.insider_customer_id = ped.insider_customer_id\n  GROUP BY 1, 2\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- TAXA DE RECOMPRA POR PRODUTO\n-- ─────────────────────────────────────────────────────────────────────────────\ntaxa_recompra AS (\n  SELECT\n    product_name,\n    COUNT(DISTINCT insider_customer_id)     AS total_compradores,\n    AVG(recomprou)                          AS metric_taxa_recompra\n  FROM recompra\n  GROUP BY product_name\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- MEDIANA DA TAXA DE RECOMPRA (referência para normalização)\n-- ─────────────────────────────────────────────────────────────────────────────\nmediana_recompra AS (\n  SELECT\n    PERCENTILE_CONT(metric_taxa_recompra, 0.5) OVER () AS mediana_taxa_recompra\n  FROM taxa_recompra\n  LIMIT 1\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- MÉDIAS DE LTV E #COMPRAS POR PRODUTO (dois recortes temporais)\n-- ─────────────────────────────────────────────────────────────────────────────\nltv_por_produto AS (\n  SELECT\n    product_name,\n    AVG(ltv_ate_compra)     AS ltv_medio_ate_primeira_compra,\n    AVG(ltv_pre_compra)     AS ltv_medio_pre_primeira_compra,\n    AVG(compras_ate_compra) AS compras_medias_ate_primeira_compra,\n    AVG(compras_pre_compra) AS compras_medias_pre_primeira_compra\n  FROM metricas_cliente_produto\n  GROUP BY product_name\n),\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- MEDIANA DO LTV POR PRODUTO (referência para normalização do ltv_ratio)\n-- ─────────────────────────────────────────────────────────────────────────────\nmediana_ltv_por_produto AS (\n  SELECT\n    PERCENTILE_CONT(ltv_medio_ate_primeira_compra, 0.5) OVER () AS mediana_ltv_produto\n  FROM ltv_por_produto\n  LIMIT 1\n)\n\n-- ─────────────────────────────────────────────────────────────────────────────\n-- OUTPUT FINAL\n-- ─────────────────────────────────────────────────────────────────────────────\nSELECT\n  tr.product_name,\n  tr.total_compradores,\n  tr.metric_taxa_recompra                                                       AS taxa_recompra_absoluta,\n  SAFE_DIVIDE(tr.metric_taxa_recompra, mr.mediana_taxa_recompra)                AS metric_taxa_recompra,\n  mr.mediana_taxa_recompra,\n\n  -- LTV temporal\n  lpp.ltv_medio_ate_primeira_compra,\n  lpp.ltv_medio_pre_primeira_compra,\n\n  -- #compras médias\n  lpp.compras_medias_ate_primeira_compra,\n  lpp.compras_medias_pre_primeira_compra,\n\n  -- referência e ratio\n  ml.mediana_ltv_produto,\n  SAFE_DIVIDE(lpp.ltv_medio_ate_primeira_compra, ml.mediana_ltv_produto) AS metric_ltv_ratio\n\nFROM taxa_recompra tr\nLEFT JOIN ltv_por_produto lpp USING (product_name)\nCROSS JOIN mediana_recompra mr\nCROSS JOIN mediana_ltv_por_produto ml\n\nORDER BY tr.metric_taxa_recompra DESC',
  'SQL_E1199E86_8709_489E_A277_2784CCE6BB51',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_recompra

,product_name,total_compradores,taxa_recompra_absoluta,metric_taxa_recompra,mediana_taxa_recompra,ltv_medio_ate_primeira_compra,ltv_medio_pre_primeira_compra,compras_medias_ate_primeira_compra,compras_medias_pre_primeira_compra,mediana_ltv_produto,metric_ltv_ratio
0,Vestido Midi de Alça FutureForm Feminino,2858,0.604269,1.388951,0.435054,1005.361795,753.715315,5.531141,4.531141,411.745763,2.441705
1,Saia Midi Kyoto Feminino,12630,0.572842,1.316716,0.435054,545.577599,254.897601,2.705859,1.705859,411.745763,1.325035
2,Shorts Kyoto Feminino,17385,0.562899,1.293860,0.435054,454.507716,257.013855,2.752200,1.752200,411.745763,1.103855
3,Camisa FutureForm Feminino,22086,0.552205,1.269279,0.435054,526.760499,267.618770,2.770488,1.770488,411.745763,1.279334
4,Bermuda Kyoto Feminino,5436,0.550589,1.265564,0.435054,704.991272,472.027010,3.965232,2.965232,411.745763,1.712200
...,...,...,...,...,...,...,...,...,...,...,...
87,Undershirt Anti Suor Gola U Masculino,2558,0.313135,0.719762,0.435054,222.183309,69.986910,1.511337,0.511337,411.745763,0.539613
88,Daily T-shirt Masculino,75708,0.311196,0.715303,0.435054,209.897616,84.249735,1.659547,0.659547,411.745763,0.509775
89,Undershirt Simples Gola V Masculino,1528,0.310209,0.713037,0.435054,241.584169,102.036376,1.749346,0.749346,411.745763,0.586731
90,Tech T-shirt Gola U Masculino,439477,0.290586,0.667932,0.435054,147.158338,24.179171,1.203207,0.203207,411.745763,0.357401


In [9]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{"sortBy":[{"id":"product_name","type":"asc"}],"filters":[],"pageSize":10,"pageIndex":0,"columnOrder":["product_name","category_4","preco_medio_venda","cmv_medio","preco_cheio_medio","markup_real","markup_entrada","metric_markup_vs_target","flag_markup"],"hiddenColumnIds":[],"columnDisplayNames":[],"conditionalFilters":[],"cellFormattingRules":[],"wrappedTextColumnIds":[]}')
else:
  _deepnote_current_table_attrs = '{"sortBy":[{"id":"product_name","type":"asc"}],"filters":[],"pageSize":10,"pageIndex":0,"columnOrder":["product_name","category_4","preco_medio_venda","cmv_medio","preco_cheio_medio","markup_real","markup_entrada","metric_markup_vs_target","flag_markup"],"hiddenColumnIds":[],"columnDisplayNames":[],"conditionalFilters":[],"cellFormattingRules":[],"wrappedTextColumnIds":[]}'

df_markup = _dntk.execute_sql(
  '-- =============================================================================\n-- 04_markup.sql\n-- Markup real praticado vs. markup target da empresa por produto\n-- Granularidade de output: product_name\n-- =============================================================================\n-- Parâmetros injetados via Python:\n--   6  → janela base (default: 8)\n--   3.85    → markup target da empresa (default: 3.85)\n-- =============================================================================\n\nWITH\n\nbase AS (\n  SELECT\n    s.product_name,\n    dre.category_4,\n\n    -- Preço médio de venda por unidade (após descontos)\n    SAFE_DIVIDE(\n      SUM(dre.revenue_after_discounts),\n      NULLIF(SUM(dre.non_refunded_quantity), 0)\n    ) AS preco_medio_venda,\n\n    -- CMV médio por unidade\n    SAFE_DIVIDE(SUM(sku_total_cost), SUM(sold_quantity)) AS cmv_medio,\n\n\n    -- Preço cheio médio cadastrado por unidade\n    SAFE_DIVIDE(\n      SUM(dre.items_potential_revenue),\n      NULLIF(SUM(dre.quantity), 0)\n    ) AS preco_cheio_medio\n\n  FROM `insider-data-lake.fpa.analytical_dre` AS dre\n  JOIN `insider-data-lake.integrated.skus` AS s USING (sku)\n\n  WHERE dre.order_status != \'Not authorized\'\n    AND LOWER(s.product_name) NOT LIKE \'%kit%\'\n    AND LOWER(s.product_name) NOT LIKE \'%defeito%\'\n    AND s.sku_state NOT IN (\'personalizacao\', \'ativo_personalizacao\')\n    AND s.sku_state = \'ativo_perene\'\n    AND dre.order_date >= DATETIME_SUB(CURRENT_DATETIME(), INTERVAL 6 MONTH)\n    -- AND dre.sold_quantity > 0\n\n  GROUP BY s.product_name, dre.category_4\n)\n\nSELECT\n  product_name,\n  category_4,\n  preco_medio_venda,\n  cmv_medio,\n  preco_cheio_medio,\n\n  -- Markup real = preço médio de venda / CMV médio\n  SAFE_DIVIDE(preco_medio_venda, NULLIF(cmv_medio, 0))       AS markup_real,\n\n  -- Markup de entrada = preço cheio / CMV (sem descontos)\n  SAFE_DIVIDE(preco_cheio_medio, NULLIF(cmv_medio, 0))       AS markup_entrada,\n\n  -- Razão vs. target (métrica do scorecard)\n  SAFE_DIVIDE(\n    SAFE_DIVIDE(preco_medio_venda, NULLIF(cmv_medio, 0)),\n    3.85\n  )                                                          AS metric_markup_vs_target,\n\n  -- Flags de anomalia\n  CASE\n    WHEN cmv_medio IS NULL OR cmv_medio = 0 THEN \'SEM_CMV\'\n    WHEN SAFE_DIVIDE(preco_medio_venda, NULLIF(cmv_medio, 0)) > 3.85 * 2 THEN \'MARKUP_ANOMALO\'\n    ELSE \'OK\'\n  END AS flag_markup\n\nFROM base\n\nORDER BY markup_real DESC',
  'SQL_E1199E86_8709_489E_A277_2784CCE6BB51',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_markup

,product_name,category_4,preco_medio_venda,cmv_medio,preco_cheio_medio,markup_real,markup_entrada,metric_markup_vs_target,flag_markup
0,High Neck Tank Feminino,Woman Casual Top T-Shirt,126.632275,23.143661,184.639892,5.471575,7.977990,1.421188,OK
1,Vestido Wingsuit Feminino,Woman Casual Dresses Dress Midi,405.823796,75.337950,575.106323,5.386711,7.633687,1.399146,OK
2,NoHo Socks,Unissex Accessory Socks,45.824455,10.520094,74.895727,4.355898,7.119302,1.131402,OK
3,NEXTECH T-shirt Masculino,Man Casual Top T-Shirt,162.168900,37.272207,205.881038,4.350934,5.523715,1.130113,OK
4,Saia Envelope Breeze Feminino,Woman Casual Bottoms Skirts Midi,224.228738,55.243653,300.406971,4.058905,5.437855,1.054261,OK
...,...,...,...,...,...,...,...,...,...
88,Cueca Slip Comfort Simples Masculino,Man Underwear Bottoms Underpants,56.738924,25.854931,74.505284,2.194511,2.881666,0.570003,OK
89,Chemise FutureForm Feminino,Woman Casual Dresses Dress Long,300.915910,145.617384,493.459547,2.066483,3.388741,0.536749,OK
90,Cueca Boxer Performance Anti Suor Masculino,Man Underwear Bottoms Underpants,70.984256,34.533526,93.667393,2.055517,2.712361,0.533901,OK
91,Cueca Boxer Comfort Simples Masculino,Man Underwear Bottoms Underpants,61.936675,30.728856,84.438189,2.015587,2.747847,0.523529,OK


In [10]:
print(f"DRE:          {len(df_dre)} produtos")
print(f"Estoque:      {len(df_estoque)} produtos")
print(f"Recompra/LTV: {len(df_recompra)} produtos")
print(f"Markup:       {len(df_markup)} produtos")

DRE:          93 produtos
Estoque:      0 produtos
Recompra/LTV: 92 produtos
Markup:       93 produtos


In [11]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_cluster = _dntk.execute_sql(
  'SELECT\n    product_name,\n    cluster\nFROM `insider-data-lake.sop_silver.portfolio_skp_clustering`',
  'SQL_E1199E86_8709_489E_A277_2784CCE6BB51',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_cluster

,product_name,cluster
0,Daily T-shirt Feminino,Long tail
1,Pack de Figurinhas INSIDER Vol. 02,Outros
2,Vestido Wingsuit Feminino,Long tail
3,Boné Sixx INSIDER + Ziraldo | Maluquinho,KILL
4,Sutiã Flex Feminino,Long tail
...,...,...
209,Regata Intech Masculino,Long tail
210,Air Blouse Feminino,KILL
211,Energy Top Feminino,KILL
212,Shorts Boxy InLounge Feminino,KILL


In [12]:
# ── Consolidar df_base ─────────────────────────────────────────────

df_base = (
    df_dre
    .merge(df_estoque,  on="product_name", how="left")
    .merge(df_recompra, on="product_name", how="left")
    .merge(df_markup,   on="product_name", how="left", suffixes=("", "_mkp"))
    .merge(df_cluster,  on="product_name", how="left")
)

# Injetar benchmarks como colunas (necessário para expressões do JSON)
df_base["benchmark_mc3"]          = SCORECARD_INPUTS["benchmark_mc3"]
df_base["benchmark_mrkup_target"] = SCORECARD_INPUTS["benchmark_mrkup_target"]

# ── Calcular metric_receita_por_skc_ativo ─────────────────────────
# Receita média por SKC ativo, normalizada pela média do cluster (Hero/Core/LT)
df_base["receita_por_skc"] = (
    df_base["receita_media_mensal"] / df_base["n_skcs_total"].replace(0, np.nan)
)
cluster_mean = df_base.groupby("cluster")["receita_por_skc"].transform("mean")
df_base["metric_receita_por_skc_ativo"] = (
    # df_base["receita_por_skc"] / cluster_mean.replace(0, np.nan)
    df_base["receita_por_skc"]
)

# ── Calcular metric_contribuicao_absoluta ─────────────────────────
# SUM(net_profit_after_marketing_costs) direto da DRE, normalizado pela mediana
# df_base["metric_contribuicao_absoluta_raw"] = df_base["contribuicao_absoluta"]
# mediana_contribuicao = df_base["metric_contribuicao_absoluta_raw"].median()
# print(f"Mediana da contribuição absoluta: {mediana_contribuicao:.2f}")
df_base["metric_contribuicao_absoluta"] = df_base["contribuicao_absoluta"]
# )

# Flags especiais (do params.json)
breakthrough_products = params.get("special_flags", {}).get("is_breakthrough", {}).get("products", [])
accessory_products    = params.get("special_flags", {}).get("is_accessory", {}).get("products", [])
df_base["is_breakthrough"] = df_base["product_name"].isin(breakthrough_products)
df_base["is_accessory"]    = df_base["product_name"].isin(accessory_products)

print(f"Base consolidada: {len(df_base)} produtos perenes")
df_base.head()

Base consolidada: 93 produtos perenes


,product_name,category_1,category_2,category_3,category_4,receita_total,unidades_total,metric_mc3,contribuicao_absoluta,metric_full_price_pct,...,metric_markup_vs_target,flag_markup,cluster,benchmark_mc3,benchmark_mrkup_target,receita_por_skc,metric_receita_por_skc_ativo,metric_contribuicao_absoluta,is_breakthrough,is_accessory
0,Tech T-shirt Gola U Masculino,Man,Man Casual,Man Casual Top,Man Casual Top T-Shirt,2.437765e+07,208586.420567,0.209256,20.461175,0.278084,...,0.751453,OK,Hero,0.2,3.85,NaN,NaN,20.461175,False,False
1,Daily T-shirt Masculino,Man,Man Casual,Man Casual Top,Man Casual Top T-Shirt,1.884322e+07,236448.262361,0.245851,16.232176,0.079197,...,0.621116,OK,Hero,0.2,3.85,NaN,NaN,16.232176,False,False
2,The Perfect Top Feminino,Woman,Woman Casual,Woman Casual Top,Woman Casual Top Tank Top,1.658082e+07,157226.651148,0.029221,2.546537,0.402459,...,0.717372,OK,Long tail,0.2,3.85,NaN,NaN,2.546537,False,False
3,Wingsuit Feminino,Woman,Woman Casual,Woman Casual Top,Woman Casual Top Jackets/Hoodies,1.015117e+07,26754.720163,0.424268,136.002011,0.635326,...,0.862647,OK,Hero,0.2,3.85,NaN,NaN,136.002011,False,False
4,Daily Light T-shirt Masculino,Man,Man Casual,Man Casual Top,Man Casual Top T-Shirt,1.005036e+07,140865.119215,0.244171,14.548545,0.999990,...,0.658219,OK,Hero,0.2,3.85,NaN,NaN,14.548545,False,False


## 2. Scorecard — Avaliação Contínua

### 2.1 Converter `params.json` para formato de pillars do motor
### 2.2 Motor de scoring (copiado do scorecard de lançamentos — não alterar)
### 2.3 Aplicar o scorecard e routing de acionáveis

In [13]:
# ── 2.1 Converter params.json → formato pillars do motor ──────────

def params_to_scorecard_config(params: dict) -> dict:
    """Converte o params.json (formato aninhado por pilar) para o formato
    de 'pillars' esperado pelo motor do scorecard de lançamentos."""

    PILLAR_DISPLAY_NAMES = {
        "unit_economics": "Unit Economics",
        "estoque": "Estoque",
        "tracao": "Tracao Comercial",
        "satisfacao": "Satisfacao e Marca",
    }

    METRIC_DISPLAY_NAMES = {
        "mc3": "MC3",
        "markup_vs_target": "Markup vs Target",
        "receita_por_skc_ativo": "Receita por SKC Ativo",
        "contribuicao_absoluta": "Contribuição Absoluta",
        "cobertura_dias": "Cobertura (dias)",
        "cv_vendas_skc": "CV Vendas entre SKCs",
        "ruptura_pct": "% Sem Ruptura",
        "disponibilidade_sku_semana": "Disponibilidade SKU×Semana",
        "share_trend": "Tendência Share no Portfólio",
        "full_price_pct": "% Full Price",
        "taxa_recompra": "Taxa de Recompra",
        "ltv_ratio": "LTV Ratio",
        "t_e_d_vs_categoria": "T&D vs Categoria",
    }

    pillars = []
    metrics = params["metrics"]
    pillar_weights = params["pillar_weights"]

    for pillar_key, pillar_weight in pillar_weights.items():
        pillar_metrics = metrics[pillar_key]
        criteria = []

        for metric_id, metric_config in pillar_metrics.items():
            if not isinstance(metric_config, dict):
                continue
            has_thresholds = "thresholds" in metric_config
            has_bands = "bands" in metric_config
            if not has_thresholds and not has_bands:
                continue

            # Skip metrics with weight 0
            if metric_config.get("weight_in_pillar", None) == 0:
                continue

            # Peso da métrica no pilar
            weight = metric_config.get("weight_in_pillar", 1.0)

            col_name = f"metric_{metric_id}"

            # Gerar regras de scoring
            if has_bands:
                bands = sorted(
                    metric_config["bands"],
                    key=lambda b: b.get("max") or float("inf"),
                )
                rules = [
                    {"op": "<=", "value": b["max"], "points": b["score"],
                     "label": f"<= {b['max']}"}
                    for b in bands if b["max"] is not None
                ]
                scoring = {
                    "missing_points": None,
                    "default_points": bands[-1]["score"],
                    "rules": rules,
                }
            elif metric_config.get("invert", False):
                th = metric_config["thresholds"]
                scoring = {
                    "missing_points": None,
                    "default_points": 0,
                    "rules": [
                        {"op": "<=", "value": th["low"],  "points": 100, "label": f"<= {th['low']}"},
                        {"op": "<=", "value": th["high"], "points": 50,  "label": f"<= {th['high']}"},
                    ],
                }
            else:
                th = metric_config["thresholds"]
                scoring = {
                    "missing_points": None,
                    "default_points": 0,
                    "rules": [
                        {"op": ">=", "value": th["high"], "points": 100, "label": f">= {th['high']}"},
                        {"op": ">=", "value": th["low"],  "points": 50,  "label": f">= {th['low']}"},
                    ],
                }

            criteria.append({
                "id": metric_id,
                "name": METRIC_DISPLAY_NAMES.get(metric_id, metric_id),
                "weight": weight,
                "calculation": {
                    "expression": col_name,
                    "reference_columns": [col_name],
                },
                "scoring": scoring,
                "null_value_handling": "redistribute",
            })

        pillars.append({
            "name": PILLAR_DISPLAY_NAMES.get(pillar_key, pillar_key),
            "weight": pillar_weight,
            "criteria": criteria,
        })

    # Classification rules
    ct = params["classification_thresholds"]
    classification_rules = [
        {"min_score": ct[label], "label": label}
        for label in ["INVEST", "KEEP", "WATCH", "AT RISK"]
    ]

    return {
        "pillars": pillars,
        "classification_rules": classification_rules,
        "options": {"normalize_weights_on_available": True},
        "id_columns": ["product_name", "category_4", "cluster"],
    }


scorecard_config = params_to_scorecard_config(scorecard_params)
print(f"Pilares: {len(scorecard_config['pillars'])}")
for p in scorecard_config["pillars"]:
    print(f"  {p['name']} (peso={p['weight']}): {len(p['criteria'])} critérios")
    for c in p["criteria"]:
        print(f"    - {c['name']} (peso={c['weight']:.2f})")

Pilares: 4
  Unit Economics (peso=0.3): 3 critérios
    - MC3 (peso=0.35)
    - Markup vs Target (peso=0.35)
    - Contribuição Absoluta (peso=0.30)
  Estoque (peso=0.25): 3 critérios
    - Cobertura (dias) (peso=0.30)
    - CV Vendas entre SKCs (peso=0.30)
    - Disponibilidade SKU×Semana (peso=0.40)
  Tracao Comercial (peso=0.2): 2 critérios
    - Tendência Share no Portfólio (peso=0.55)
    - % Full Price (peso=0.45)
  Satisfacao e Marca (peso=0.25): 3 critérios
    - Taxa de Recompra (peso=0.25)
    - LTV Ratio (peso=0.15)
    - T&D vs Categoria (peso=0.60)


In [14]:
# ══════════════════════════════════════════════════════════════════
# 2.2 Motor de Scoring
# Copiado integralmente do scorecard de lançamentos — NÃO ALTERAR
# ══════════════════════════════════════════════════════════════════


# ── Funções auxiliares ────────────────────────────────────────────

def slugify(text: str) -> str:
    """Converte texto em slug seguro para nomes de coluna."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def ensure_series(obj, index: pd.Index) -> pd.Series:
    """Converte escalares ou arrays em Series com o mesmo index do df."""
    if isinstance(obj, pd.Series):
        return obj.reindex(index)
    if np.isscalar(obj):
        return pd.Series(obj, index=index)
    return pd.Series(obj, index=index)


def safe_div(numerador, denominador):
    """Divisão segura para Series, com tratamento de zero e infinito."""
    idx = numerador.index if isinstance(numerador, pd.Series) else None
    if idx is None and isinstance(denominador, pd.Series):
        idx = denominador.index
    if idx is None:
        raise ValueError("safe_div precisa de pelo menos um argumento em formato Series.")

    num = pd.to_numeric(ensure_series(numerador, idx), errors="coerce")
    den = pd.to_numeric(ensure_series(denominador, idx), errors="coerce").replace(0, np.nan)
    out = num / den
    return out.replace([np.inf, -np.inf], np.nan)


# ── Validação ─────────────────────────────────────────────────────

def validate_config(df: pd.DataFrame, config: dict) -> None:
    """Valida a estrutura mínima do JSON e as colunas de referência."""
    if "pillars" not in config:
        raise ValueError("A configuração precisa da chave 'pillars'.")

    valid_null_handling = {"redistribute", "penalize"}

    missing_columns = []
    for pillar in config["pillars"]:
        if "criteria" not in pillar:
            raise ValueError(f"O pilar {pillar.get('name')} precisa da chave 'criteria'.")

        null_score = pillar.get("all_criteria_null_score")
        if null_score is not None and not isinstance(null_score, (int, float)):
            raise ValueError(
                f"O pilar {pillar.get('name')} tem all_criteria_null_score inválido."
            )

        for criterion in pillar["criteria"]:
            calc = criterion.get("calculation", {})
            expr = calc.get("expression")
            if not expr:
                raise ValueError(
                    f"O critério {criterion.get('id')} precisa de 'calculation.expression'."
                )

            handling = criterion.get("null_value_handling", "redistribute")
            if handling not in valid_null_handling:
                raise ValueError(
                    f"O critério {criterion.get('id')} tem null_value_handling='{handling}' "
                    f"inválido. Valores aceitos: {valid_null_handling}"
                )

            for col in calc.get("reference_columns", []):
                if col not in df.columns:
                    missing_columns.append((criterion.get("id"), col))

            if "scoring" not in criterion:
                raise ValueError(f"O critério {criterion.get('id')} precisa de 'scoring'.")

    if missing_columns:
        msg = "\n".join(
            [f"- critério '{crit}': coluna ausente '{col}'" for crit, col in missing_columns]
        )
        raise KeyError("Colunas faltantes no df_base:\n" + msg)


# ── Avaliação de expressões ───────────────────────────────────────

def evaluate_expression(df: pd.DataFrame, criterion: dict, inputs: dict = None) -> pd.Series:
    """Avalia a fórmula do critério usando colunas do df e inputs externos."""
    calc = criterion["calculation"]
    expr = calc["expression"]

    local_env = {col: df[col] for col in df.columns}
    if inputs:
        local_env.update(inputs)
    local_env.update({"np": np, "pd": pd, "safe_div": safe_div})

    result = eval(expr, {"__builtins__": {}}, local_env)
    result = ensure_series(result, df.index)
    result = pd.to_numeric(result, errors="coerce")
    return result.replace([np.inf, -np.inf], np.nan)


# ── Scoring ───────────────────────────────────────────────────────

def compare_series(values: pd.Series, op: str, target) -> pd.Series:
    """Aplica um comparador entre a série e um alvo."""
    ops = {
        ">=": lambda v, t: v >= t,
        ">": lambda v, t: v > t,
        "<=": lambda v, t: v <= t,
        "<": lambda v, t: v < t,
        "==": lambda v, t: v == t,
        "!=": lambda v, t: v != t,
    }
    if op == "between":
        lower, upper = target
        return values.between(lower, upper, inclusive="both")
    if op in ops:
        return ops[op](values, target)
    raise ValueError(f"Operador não suportado: {op}")


def apply_scoring_rules(values: pd.Series, scoring: dict):
    """Aplica as faixas de score na ordem em que aparecem no JSON."""
    default_points = scoring.get("default_points", 0)
    missing_points = scoring.get("missing_points", np.nan)
    default_label = scoring.get("default_label", "default")

    scores = pd.Series(np.nan, index=values.index, dtype="float64")
    labels = pd.Series(pd.NA, index=values.index, dtype="object")

    missing_mask = values.isna()
    scores.loc[missing_mask] = missing_points
    labels.loc[missing_mask] = "missing"

    remaining = values.notna().copy()
    for i, rule in enumerate(scoring.get("rules", []), start=1):
        op = rule["op"]
        target = rule["value"]
        mask = remaining & compare_series(values, op, target)
        if mask.any():
            scores.loc[mask] = rule["points"]
            labels.loc[mask] = rule.get("label", f"regra_{i}")
            remaining.loc[mask] = False

    scores.loc[remaining] = default_points
    labels.loc[remaining] = default_label
    return scores, labels


def resolve_null_scores(
    values: pd.Series, scores: pd.Series, labels: pd.Series, handling: str
 ) -> tuple:
    """Aplica a política de tratamento de nulos do critério.

    - 'redistribute': força score=NaN → peso redistribuído aos demais
    - 'penalize': mantém missing_points como score
    """
    if handling == "redistribute":
        null_mask = values.isna()
        scores = scores.copy()
        labels = labels.copy()
        scores.loc[null_mask] = np.nan
        labels.loc[null_mask] = "redistributed"
    return scores, labels


# ── Média ponderada ───────────────────────────────────────────────

def weighted_average_from_columns(
    df: pd.DataFrame,
    col_weights: dict,
    normalize_available: bool = True,
 ) -> pd.Series:
    """Calcula média ponderada a partir de colunas já calculadas no df."""
    if not col_weights:
        return pd.Series(np.nan, index=df.index)

    weights = pd.Series(col_weights, dtype="float64")
    weights = weights / weights.sum()

    if normalize_available:
        weighted_sum = pd.Series(0.0, index=df.index)
        weight_sum = pd.Series(0.0, index=df.index)

        for col, weight in weights.items():
            valid = df[col].notna().astype(float)
            weighted_sum = weighted_sum + df[col].fillna(0) * weight
            weight_sum = weight_sum + valid * weight

        return weighted_sum / weight_sum.replace(0, np.nan)

    out = pd.Series(0.0, index=df.index)
    for col, weight in weights.items():
        out = out + df[col].fillna(0) * weight
    return out


# ── Classificação ─────────────────────────────────────────────────

def classify_scores(score_series: pd.Series, classification_rules: list) -> pd.Series:
    """Classifica o score total conforme as regras de corte."""
    rules = sorted(classification_rules, key=lambda x: x["min_score"], reverse=True)
    labels = pd.Series(pd.NA, index=score_series.index, dtype="object")

    remaining = score_series.notna().copy()
    for rule in rules:
        mask = remaining & (score_series >= rule["min_score"])
        labels.loc[mask] = rule["label"]
        remaining.loc[mask] = False

    return labels


# ── Orquestrador principal ────────────────────────────────────────

def apply_scorecard(df: pd.DataFrame, config: dict, inputs: dict = None):
    """Aplica o scorecard ao df_base e devolve (df_resultado, df_memorial)."""
    validate_config(df, config)

    result = df.copy()
    memorial_frames = []

    normalize_available = config.get("options", {}).get("normalize_weights_on_available", True)
    id_columns = [c for c in config.get("id_columns", []) if c in result.columns]
    if not id_columns:
        id_columns = [
            c
            for c in ["product_name", "category_4", "sku_state", "data_lancamento"]
            if c in result.columns
        ]

    pillar_score_col_weights = {}
    for pillar in config["pillars"]:
        pillar_name = pillar["name"]
        pillar_slug = slugify(pillar_name)
        pillar_weight = pillar.get("weight", 1)
        criteria = pillar.get("criteria", [])

        crit_weights = {
            criterion["id"]: float(criterion.get("weight", 1)) for criterion in criteria
        }
        crit_weight_total = sum(crit_weights.values()) or 1.0
        crit_weights = {k: v / crit_weight_total for k, v in crit_weights.items()}

        score_col_weights = {}
        for criterion in criteria:
            crit_id = criterion["id"]
            crit_slug = slugify(crit_id)
            handling = criterion.get("null_value_handling", "redistribute")

            metric_col = f"valor__{pillar_slug}__{crit_slug}"
            score_col = f"score__{pillar_slug}__{crit_slug}"
            faixa_col = f"faixa__{pillar_slug}__{crit_slug}"

            values = evaluate_expression(result, criterion, inputs)
            scores, labels = apply_scoring_rules(values, criterion["scoring"])
            scores, labels = resolve_null_scores(values, scores, labels, handling)

            result[metric_col] = values
            result[score_col] = scores
            result[faixa_col] = labels
            score_col_weights[score_col] = crit_weights[crit_id]

            memorial = result[id_columns].copy()
            memorial["pilar"] = pillar_name
            memorial["peso_pilar"] = pillar_weight
            memorial["criterio_id"] = crit_id
            memorial["criterio"] = criterion["name"]
            memorial["peso_criterio_no_pilar"] = crit_weights[crit_id]
            memorial["null_value_handling"] = handling
            memorial["memoria_calculo"] = criterion.get("memoria_calculo", "")
            memorial["expressao"] = criterion["calculation"]["expression"]
            memorial["colunas_referencia"] = ", ".join(
                criterion["calculation"].get("reference_columns", [])
            )
            memorial["inputs"] = json.dumps(inputs or {}, ensure_ascii=False)
            memorial["valor_calculado"] = values.values
            memorial["score_criterio"] = scores.values
            memorial["faixa_aplicada"] = labels.values
            memorial_frames.append(memorial)

        pillar_score_col = f"score_pilar__{pillar_slug}"
        result[pillar_score_col] = weighted_average_from_columns(
            result, score_col_weights, normalize_available=normalize_available,
        )

        all_null_score = pillar.get("all_criteria_null_score")
        if all_null_score is not None:
            result[pillar_score_col] = result[pillar_score_col].fillna(all_null_score)

        pillar_score_col_weights[pillar_score_col] = pillar_weight

    result["score_total"] = weighted_average_from_columns(
        result, pillar_score_col_weights, normalize_available=normalize_available,
    )
    result["classificacao"] = classify_scores(
        result["score_total"], config["classification_rules"]
    )

    df_memorial = pd.concat(memorial_frames, axis=0, ignore_index=True)
    return result, df_memorial


# ── Tabela de configuração para revisão ───────────────────────────

def config_to_dataframe(config: dict, inputs: dict = None) -> pd.DataFrame:
    """Transforma a configuração do scorecard em tabela para revisão rápida."""
    rows = []
    for pillar in config["pillars"]:
        for criterion in pillar["criteria"]:
            rows.append({
                "pilar": pillar["name"],
                "peso_pilar": pillar.get("weight", 1),
                "criterio_id": criterion["id"],
                "criterio": criterion["name"],
                "peso_criterio": criterion.get("weight", 1),
                "memoria_calculo": criterion.get("memoria_calculo", ""),
                "expressao": criterion["calculation"]["expression"],
                "colunas_referencia": ", ".join(
                    criterion["calculation"].get("reference_columns", [])
                ),
                "regras_score": json.dumps(
                    criterion["scoring"].get("rules", []), ensure_ascii=False
                ),
                "default_points": criterion["scoring"].get("default_points"),
                "missing_points": criterion["scoring"].get("missing_points"),
            })
    return pd.DataFrame(rows)

In [15]:
# ── 2.2 Preview da configuração gerada ────────────────────────────

df_config = config_to_dataframe(scorecard_config, SCORECARD_INPUTS)
display(df_config)

,pilar,peso_pilar,criterio_id,criterio,peso_criterio,memoria_calculo,expressao,colunas_referencia,regras_score,default_points,missing_points
0,Unit Economics,0.30,mc3,MC3,0.35,,metric_mc3,metric_mc3,"[{""op"": ""<="", ""value"": 0.13, ""points"": 0, ""lab...",100,None
1,Unit Economics,0.30,markup_vs_target,Markup vs Target,0.35,,metric_markup_vs_target,metric_markup_vs_target,"[{""op"": ""<="", ""value"": 0.75, ""points"": 0, ""lab...",100,None
2,Unit Economics,0.30,contribuicao_absoluta,Contribuição Absoluta,0.30,,metric_contribuicao_absoluta,metric_contribuicao_absoluta,"[{""op"": ""<="", ""value"": 10, ""points"": 0, ""label...",100,None
3,Estoque,0.25,cobertura_dias,Cobertura (dias),0.30,,metric_cobertura_dias,metric_cobertura_dias,"[{""op"": ""<="", ""value"": 60, ""points"": 100, ""lab...",0,None
4,Estoque,0.25,cv_vendas_skc,CV Vendas entre SKCs,0.30,,metric_cv_vendas_skc,metric_cv_vendas_skc,"[{""op"": ""<="", ""value"": 0.2, ""points"": 100, ""la...",0,None
5,Estoque,0.25,disponibilidade_sku_semana,Disponibilidade SKU×Semana,0.40,,metric_disponibilidade_sku_semana,metric_disponibilidade_sku_semana,"[{""op"": ""<="", ""value"": 0.6, ""points"": 0, ""labe...",100,None
6,Tracao Comercial,0.20,share_trend,Tendência Share no Portfólio,0.55,,metric_share_trend,metric_share_trend,"[{""op"": ""<="", ""value"": -3, ""points"": 0, ""label...",100,None
7,Tracao Comercial,0.20,full_price_pct,% Full Price,0.45,,metric_full_price_pct,metric_full_price_pct,"[{""op"": ""<="", ""value"": 0.5, ""points"": 0, ""labe...",100,None
8,Satisfacao e Marca,0.25,taxa_recompra,Taxa de Recompra,0.25,,metric_taxa_recompra,metric_taxa_recompra,"[{""op"": ""<="", ""value"": 0.85, ""points"": 0, ""lab...",100,None
9,Satisfacao e Marca,0.25,ltv_ratio,LTV Ratio,0.15,,metric_ltv_ratio,metric_ltv_ratio,"[{""op"": ""<="", ""value"": 0.85, ""points"": 0, ""lab...",100,None


In [16]:
# ── 2.3 Aplicar o scorecard ────────────────────────────────────────

df_scorecard, df_memorial = apply_scorecard(df_base, scorecard_config, SCORECARD_INPUTS)

score_cols_preview = ["classificacao"] + [
    c for c in df_scorecard.columns if c.startswith("score_pilar__")
] + ["score_total"]

id_cols_preview = [
    c for c in ["product_name", "category_4", "cluster"]
    if c in df_scorecard.columns
]

df_scorecard[id_cols_preview + score_cols_preview]

,product_name,category_4,cluster,classificacao,score_pilar__unit_economics,score_pilar__estoque,score_pilar__tracao_comercial,score_pilar__satisfacao_e_marca,score_total
0,Tech T-shirt Gola U Masculino,Man Casual Top T-Shirt,Hero,WATCH,33.75,NaN,27.50,60.00,40.833333
1,Daily T-shirt Masculino,Man Casual Top T-Shirt,Hero,WATCH,33.75,NaN,27.50,60.00,40.833333
2,The Perfect Top Feminino,Woman Casual Top Tank Top,Long tail,AT RISK,0.00,NaN,27.50,45.00,22.333333
3,Wingsuit Feminino,Woman Casual Top Jackets/Hoodies,Hero,KEEP,82.50,NaN,50.00,71.25,70.083333
4,Daily Light T-shirt Masculino,Man Casual Top T-Shirt,Hero,WATCH,33.75,NaN,45.00,50.00,42.166667
...,...,...,...,...,...,...,...,...,...
88,Regata Nadador IN-ACTION Seamless Feminino,Woman Fitness/Sportswear Top Tank Top,Long tail,KEEP,76.25,NaN,45.00,63.75,63.750000
89,Undershirt Anti Suor Gola U Masculino,Man Underwear Top Undershirt,Long tail,WATCH,41.25,NaN,61.25,15.00,37.833333
90,Undershirt Simples Gola V Masculino,Man Underwear Top Undershirt,Long tail,WATCH,25.00,NaN,72.50,60.00,49.333333
91,Pochete Slim Esportiva SprintIn,Unissex Accessory Bag Pochete,Long tail,KEEP,7.50,NaN,100.00,85.00,58.000000


In [17]:
# ── 2.4 Routing de acionáveis ──────────────────────────────────────

pillar_score_cols = [c for c in df_scorecard.columns if c.startswith("score_pilar__")]

df_scorecard["pilar_mais_baixo"] = (
    df_scorecard[pillar_score_cols]
    .idxmin(axis=1)
    .str.replace("score_pilar__", "")
)

df_scorecard["responsavel_acionavel"] = (
    df_scorecard["pilar_mais_baixo"].map(PILAR_RESPONSAVEL)
)

df_scorecard[["product_name", "classificacao", "score_total", "pilar_mais_baixo", "responsavel_acionavel"]].head(10)

,product_name,classificacao,score_total,pilar_mais_baixo,responsavel_acionavel
0,Tech T-shirt Gola U Masculino,WATCH,40.833333,tracao_comercial,Growth/Mídia
1,Daily T-shirt Masculino,WATCH,40.833333,tracao_comercial,Growth/Mídia
2,The Perfect Top Feminino,AT RISK,22.333333,unit_economics,Revenue Management
3,Wingsuit Feminino,KEEP,70.083333,tracao_comercial,Growth/Mídia
4,Daily Light T-shirt Masculino,WATCH,42.166667,unit_economics,Revenue Management
5,Core T-shirt Masculino,WATCH,33.000000,satisfacao_e_marca,Produto Físico
6,Maxi Saia NYIN Feminino,KEEP,52.333333,tracao_comercial,Growth/Mídia
7,Calça FutureForm Masculino,KEEP,68.833333,tracao_comercial,Growth/Mídia
8,Camisa FutureForm Feminino,KEEP,68.166667,tracao_comercial,Growth/Mídia
9,Calça FutureForm Feminino,KEEP,62.666667,tracao_comercial,Growth/Mídia


In [18]:
# ── 2.5 Tier-based pillar weights (recomputa score_total por cluster) ──

tier_weights = params.get("tier_weights", {})
if tier_weights and "cluster" in df_scorecard.columns:
    pillar_slug_map = {
        "unit_economics": "unit_economics",
        "estoque": "estoque",
        "tracao": "tracao_comercial",
        "satisfacao": "satisfacao_e_marca",
    }
    for tier, weights in tier_weights.items():
        mask = df_scorecard["cluster"] == tier
        if not mask.any():
            continue
        total_w = sum(weights.values())
        new_score = sum(
            df_scorecard.loc[mask, f"score_pilar__{pillar_slug_map[k]}"] * (v / total_w)
            for k, v in weights.items()
        )
        df_scorecard.loc[mask, "score_total"] = new_score

    # Re-classificar após ajuste de tier
    ct = params["classification_thresholds"]
    sorted_classes = sorted(ct.items(), key=lambda x: -x[1])
    df_scorecard["classificacao"] = df_scorecard["score_total"].apply(
        lambda s: next((k for k, v in sorted_classes if s >= v), "AT RISK")
    )
    print("✓ Tier-based weights aplicados:", {t: mask.sum() for t, mask in
          [(t, df_scorecard["cluster"] == t) for t in tier_weights.keys()]})

✓ Tier-based weights aplicados: {'Hero': 18, 'Core': 16, 'Long tail': 54}


In [19]:
# ── 2.6 FVET: Veto rules — override de classificação para thresholds críticos ──

fvet_rules = params.get("fvet_rules", [])
df_scorecard["fvet_override"] = False
df_scorecard["fvet_reason"] = ""

FVET_COMPARATORS = {
    "less_than": lambda a, b: a < b,
    "greater_than": lambda a, b: a > b,
    "less_equal": lambda a, b: a <= b,
    "greater_equal": lambda a, b: a >= b,
}

for rule in fvet_rules:
    col = rule["column"]
    if col not in df_scorecard.columns:
        print(f"⚠ FVET: coluna '{col}' não encontrada — regra ignorada")
        continue
    compare_fn = FVET_COMPARATORS.get(rule["operator"])
    if compare_fn is None:
        continue
    mask = compare_fn(df_scorecard[col], rule["threshold"])
    override_mask = mask & ~df_scorecard["fvet_override"]
    df_scorecard.loc[override_mask, "classificacao"] = rule["override_classification"]
    df_scorecard.loc[override_mask, "fvet_override"] = True
    df_scorecard.loc[override_mask, "fvet_reason"] = rule["reason"]

n_fvet = df_scorecard["fvet_override"].sum()
print(f"✓ FVET: {n_fvet} produto(s) com veto aplicado")
if n_fvet > 0:
    display(df_scorecard.loc[df_scorecard["fvet_override"],
            ["product_name", "score_total", "classificacao", "fvet_reason"]])

✓ FVET: 1 produto(s) com veto aplicado


,product_name,score_total,classificacao,fvet_reason
27,Tech T-shirt Heavy Slim Masculino,NaN,AT RISK,FVET: Satisfação crítica — veto automático


In [20]:
# ── 2.7 MC3 fragility flag + Health metrics + Diagnostics ─────────

# MC3 fragility flag
if params.get("fragility_flags", {}).get("mc3", {}).get("active"):
    df_scorecard["flag_mc3_atribuicao_fragil"] = True

# Health metrics layer (informativo, sem impacto na nota)
health_cols_config = params.get("health_metrics", {})
for metric_name, config in health_cols_config.items():
    src = config["source_column"]
    if src in df_scorecard.columns:
        df_scorecard[f"health__{metric_name}"] = df_scorecard[src]
    elif src in df_base.columns:
        df_scorecard[f"health__{metric_name}"] = df_base[src].values

# Contribuição absoluta raw como health metric
if "metric_contribuicao_absoluta_raw" in df_base.columns:
    df_scorecard["health__contribuicao_absoluta_raw"] = df_base["metric_contribuicao_absoluta_raw"].values

# Correlation diagnostic
if "metric_receita_por_skc_ativo" in df_base.columns and "metric_cv_vendas_skc" in df_base.columns:
    corr = df_base[["metric_receita_por_skc_ativo", "metric_cv_vendas_skc"]].corr().iloc[0, 1]
    print(f"Correlação receita_por_skc_ativo vs cv_vendas_skc: {corr:.3f}")
    if abs(corr) > 0.85:
        print("⚠  Correlação > 0.85 — avaliar se vale manter ambas as métricas")

# Rating stub
print("metric_rating: stub — aguardando validação de fonte de dados (P2)")

print(f"\n✓ Health metrics adicionadas: {[c for c in df_scorecard.columns if c.startswith('health__')]}")

Correlação receita_por_skc_ativo vs cv_vendas_skc: nan
metric_rating: stub — aguardando validação de fonte de dados (P2)

✓ Health metrics adicionadas: ['health__receita_absoluta_mensal', 'health__percentil_receita']


## 3. Checklist Operacional

Avaliação binária (passa / não passa) para cada produto.
Ao contrário do scorecard (nota contínua), aqui cada critério tem threshold fixo.

Os critérios são definidos em `CHECKLIST_CRITERIA` na célula de setup.

In [21]:
# ── Comparadores suportados ───────────────────────────────────────
_COMPARATORS = {
    "greater_equal": lambda val, ref: val >= ref,
    "less_equal": lambda val, ref: val <= ref,
    "equal": lambda val, ref: val == ref,
}


def apply_checklist(df: pd.DataFrame, criteria: dict) -> pd.DataFrame:
    """Avalia os critérios binários do checklist para cada produto.

    Retorna um DataFrame com uma coluna booleana e uma coluna de valor
    para cada subcritério.
    """
    records = []
    for produto in df["product_name"].unique():
        row = df[df["product_name"] == produto].iloc[0]
        result = {"Produto": produto}
        for grupo, subcrit in criteria.items():
            for nome, spec in subcrit.items():
                key = f"{grupo} - {nome}"
                val = row.get(spec["column"])
                comparator = _COMPARATORS[spec["comparison"]]
                result[key] = bool(comparator(val, spec["check_value"])) if pd.notna(val) else False
                result[f"{key} value"] = val
        records.append(result)
    return pd.DataFrame(records)


df_checklist = apply_checklist(df_scorecard, CHECKLIST_CRITERIA)
print(f"Checklist: {len(df_checklist)} produtos avaliados")
df_checklist

Checklist: 93 produtos avaliados


,Produto,Unit Economics - MC3 mínimo,Unit Economics - MC3 mínimo value,Unit Economics - Markup target,Unit Economics - Markup target value,Saúde de Estoque - Cobertura < 180d,Saúde de Estoque - Cobertura < 180d value,Saúde de Estoque - CV vendas SKC < 1.2,Saúde de Estoque - CV vendas SKC < 1.2 value,Saúde de Estoque - Disponibilidade > 70%,Saúde de Estoque - Disponibilidade > 70% value,Satisfação do Cliente - T&D ≤ 1.5x categoria,Satisfação do Cliente - T&D ≤ 1.5x categoria value
0,Tech T-shirt Gola U Masculino,True,0.209256,False,0.751453,False,NaN,False,NaN,False,NaN,True,0.897223
1,Daily T-shirt Masculino,True,0.245851,False,0.621116,False,NaN,False,NaN,False,NaN,True,0.851465
2,The Perfect Top Feminino,False,0.029221,False,0.717372,False,NaN,False,NaN,False,NaN,True,0.927322
3,Wingsuit Feminino,True,0.424268,True,0.862647,False,NaN,False,NaN,False,NaN,True,1.000000
4,Daily Light T-shirt Masculino,True,0.244171,False,0.658219,False,NaN,False,NaN,False,NaN,True,1.153017
...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,Regata Nadador IN-ACTION Seamless Feminino,True,0.240415,True,0.957761,False,NaN,False,NaN,False,NaN,True,1.181360
89,Undershirt Anti Suor Gola U Masculino,True,0.250952,False,0.602183,False,NaN,False,NaN,False,NaN,True,1.296823
90,Undershirt Simples Gola V Masculino,True,0.202667,False,0.655649,False,NaN,False,NaN,False,NaN,True,0.862612
91,Pochete Slim Esportiva SprintIn,False,0.114655,False,0.625854,False,NaN,False,NaN,False,NaN,True,1.000000


## 4. Análises de Negócio

Visualizações reportáveis para os times de Produto, Growth e Revenue Management.

In [22]:
# ── 4.1 Distribuição por Classificação (donut chart) ──────────────

class_counts = df_scorecard["classificacao"].value_counts().reset_index()
class_counts.columns = ["classificacao", "count"]

fig_41 = px.pie(
    class_counts,
    values="count",
    names="classificacao",
    hole=0.45,
    color="classificacao",
    color_discrete_map=COLOR_MAP_CLASSIFICACAO,
    title="Distribuição de Produtos por Classificação",
)
fig_41.update_traces(textinfo="percent+value")
fig_41.show()

In [23]:
# ── 4.2 Score Médio por Categoria ─────────────────────────────────

if "category_4" in df_scorecard.columns:
    score_by_cat = (
        df_scorecard.groupby(["category_1", "category_4"])["score_total"]
        .agg(["mean", "count"])
        .reset_index()
        .sort_values("mean", ascending=True)
    )

    fig_42 = px.bar(
        score_by_cat,
        x="mean",
        y="category_4",
        color='category_1',
        orientation="h",
        text="count",
        title="Score Médio por Categoria (category_4)",
        labels={"mean": "Score Médio", "category_4": "Categoria", "count": "Qtd Produtos"},
    )
    fig_42.update_traces(texttemplate="%{text} produtos", textposition="outside")
    fig_42.update_layout(xaxis_range=[0, 100], height=300 + 10 * len(score_by_cat), margin=dict(l=150, r=50, t=50, b=50))
    fig_42.show()

In [24]:
# ── 4.3 Quadrante: Unit Economics × Satisfação & Marca ────────────

ue_col = [c for c in df_scorecard.columns if c.startswith("score_pilar__unit")]
sat_col = [c for c in df_scorecard.columns if c.startswith("score_pilar__satisf")]

if ue_col and sat_col:
    df_quad = df_scorecard.dropna(subset=[ue_col[0], sat_col[0]]).copy()
    
    df_quad['size_metric'] = df_quad["receita_media_mensal"]**0.5

    fig_43 = px.scatter(
        df_quad,
        x=ue_col[0],
        y=sat_col[0],
        size="size_metric",
        color="classificacao",
        color_discrete_map=COLOR_MAP_CLASSIFICACAO,
        hover_name="product_name",
        title="Quadrante: Unit Economics × Satisfação & Marca",
        labels={ue_col[0]: "Score Unit Economics", sat_col[0]: "Score Satisfação & Marca"},
    )
    fig_43.add_hline(y=50, line_dash="dash", line_color="gray", opacity=0.5)
    fig_43.add_vline(x=50, line_dash="dash", line_color="gray", opacity=0.5)
    fig_43.update_layout(xaxis_range=[0, 100], yaxis_range=[0, 100])
    fig_43.show()

In [25]:
# ── 4.3c Scatter + Violin de Preços por Quadrante (Recompra × LTV) ──
from plotly.subplots import make_subplots

if "metric_taxa_recompra" in df_scorecard.columns and "ltv_medio_ate_primeira_compra" in df_scorecard.columns:
    df_combined = (
        df_scorecard
        .loc[df_scorecard["product_name"] != "Insider Wash Bag"]
        .dropna(subset=["metric_taxa_recompra", "ltv_medio_ate_primeira_compra", "preco_medio_venda"])
        .copy()
    )

    # Referências: mesmas usadas no scatter 4.3b
    media_recompra = df_scorecard["metric_taxa_recompra"].dropna().mean()
    mediana_ltv = df_scorecard["ltv_medio_ate_primeira_compra"].dropna().median()

    # Classificar em quadrantes
    def _quadrante(row):
        alto_recompra = row["metric_taxa_recompra"] >= media_recompra
        alto_ltv = row["ltv_medio_ate_primeira_compra"] >= mediana_ltv
        if alto_recompra and alto_ltv:
            return "↗ Alta Recompra + Alto LTV"
        elif alto_recompra and not alto_ltv:
            return "↘ Alta Recompra + Baixo LTV"
        elif not alto_recompra and alto_ltv:
            return "↖ Baixa Recompra + Alto LTV"
        else:
            return "↙ Baixa Recompra + Baixo LTV"

    df_combined["quadrante"] = df_combined.apply(_quadrante, axis=1)

    quad_order = [
        "↗ Alta Recompra + Alto LTV",
        "↖ Baixa Recompra + Alto LTV",
        "↘ Alta Recompra + Baixo LTV",
        "↙ Baixa Recompra + Baixo LTV",
    ]
    COLOR_MAP_QUAD = {
        "↗ Alta Recompra + Alto LTV": "#2ecc71",
        "↖ Baixa Recompra + Alto LTV": "#f39c12",
        "↘ Alta Recompra + Baixo LTV": "#3498db",
        "↙ Baixa Recompra + Baixo LTV": "#e74c3c",
    }

    # Layout: scatter (main) | violin preço (direita) | violin preço (topo)
    fig_combined = make_subplots(
        rows=2, cols=2,
        column_widths=[0.75, 0.25],
        row_heights=[0.25, 0.75],
        shared_xaxes=True,
        shared_yaxes=True,
        horizontal_spacing=0.02,
        vertical_spacing=0.02,
        specs=[
            [{"type": "xy"}, {"type": "xy"}],   # topo: dist preço no eixo X | vazio
            [{"type": "xy"}, {"type": "xy"}],   # scatter | dist preço no eixo Y
        ],
    )

    # ── Scatter principal (row=2, col=1) ──
    for q in quad_order:
        mask = df_combined["quadrante"] == q
        df_q = df_combined[mask]
        if len(df_q) == 0:
            continue
        fig_combined.add_trace(
            go.Scatter(
                x=df_q["metric_taxa_recompra"],
                y=df_q["ltv_medio_ate_primeira_compra"],
                mode="markers",
                marker=dict(
                    size=df_q["preco_medio_venda"].clip(50, 600).map(lambda p: 5 + (p - 50) / 600 * 20),
                    color=COLOR_MAP_QUAD[q],
                    opacity=0.7,
                    line=dict(width=0.5, color="white"),
                ),
                name=q,
                text=df_q["product_name"],
                customdata=np.stack([
                    df_q["preco_medio_venda"],
                    df_q["category_4"],
                ], axis=-1),
                hovertemplate=(
                    "<b>%{text}</b><br>"
                    "Recompra: %{x:.1f}<br>"
                    "LTV até 1ª: R$ %{y:.0f}<br>"
                    "Preço: R$ %{customdata[0]:.0f}<br>"
                    "Cat4: %{customdata[1]}<extra></extra>"
                ),
                legendgroup=q,
            ),
            row=2, col=1,
        )

    # Linhas de referência no scatter
    fig_combined.add_vline(x=media_recompra, line_dash="dash", line_color="gray", opacity=0.6, row=2, col=1)
    fig_combined.add_hline(y=mediana_ltv, line_dash="dash", line_color="gray", opacity=0.6, row=2, col=1)

    # ── Violin de preço por quadrante (direita, row=2, col=2) ──
    for q in quad_order:
        mask = df_combined["quadrante"] == q
        df_q = df_combined[mask]
        if len(df_q) == 0:
            continue
        fig_combined.add_trace(
            go.Violin(
                y=df_q["ltv_medio_ate_primeira_compra"],
                x=[q.split(" ")[0]] * len(df_q),  # usa emoji como label curto
                marker_color=COLOR_MAP_QUAD[q],
                fillcolor=COLOR_MAP_QUAD[q],
                opacity=0.5,
                line_color=COLOR_MAP_QUAD[q],
                box_visible=True,
                meanline_visible=True,
                name=q,
                legendgroup=q,
                showlegend=False,
                side="positive",
                scalemode="width",
                width=0.8,
                customdata=df_q["preco_medio_venda"].values,
                hovertemplate="LTV: R$ %{y:.0f}<br>Preço: R$ %{customdata:.0f}<extra></extra>",
            ),
            row=2, col=2,
        )

    # ── Violin/Box de preço (topo, row=1, col=1) — distribuição de preço por quadrante ao longo do eixo X ──
    for q in quad_order:
        mask = df_combined["quadrante"] == q
        df_q = df_combined[mask]
        if len(df_q) == 0:
            continue
        fig_combined.add_trace(
            go.Box(
                x=df_q["metric_taxa_recompra"],
                marker_color=COLOR_MAP_QUAD[q],
                fillcolor=COLOR_MAP_QUAD[q],
                opacity=0.4,
                line_color=COLOR_MAP_QUAD[q],
                name=q,
                legendgroup=q,
                showlegend=False,
                boxpoints=False,
            ),
            row=1, col=1,
        )

    fig_combined.update_layout(
        height=750,
        width=1000,
        title=dict(
            text=(
                "Recompra × LTV com distribuição marginal por quadrante<br>"
                f"<span style='font-size:10px;color:gray'>"
                f"Cortes: Recompra = {media_recompra:.1%} (média) | LTV = R$ {mediana_ltv:.0f} (mediana) | "
                f"Tamanho = Preço Médio</span>"
            ),
        ),
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5, font_size=9),
    )

    # Eixos do scatter (row=2, col=1)
    fig_combined.update_xaxes(title_text="Taxa de Recompra", tickformat=".0%", row=2, col=1)
    fig_combined.update_yaxes(title_text="LTV Médio até 1ª Compra (R$)", tickprefix="R$ ", row=2, col=1)

    # Esconde eixos dos marginais
    fig_combined.update_xaxes(showticklabels=False, row=1, col=1)
    fig_combined.update_yaxes(showticklabels=False, row=1, col=1)
    fig_combined.update_xaxes(showticklabels=False, row=2, col=2)
    fig_combined.update_yaxes(showticklabels=False, row=2, col=2)
    fig_combined.update_xaxes(showticklabels=False, row=1, col=2)
    fig_combined.update_yaxes(showticklabels=False, row=1, col=2)

    fig_combined.show()

    # Resumo tabular
    resumo_quad = (
        df_combined.groupby("quadrante")["preco_medio_venda"]
        .agg(["count", "mean", "median", "min", "max"])
        .reindex(quad_order)
        .round(2)
    )
    resumo_quad.columns = ["n_produtos", "preco_medio", "preco_mediana", "preco_min", "preco_max"]
    display(resumo_quad)

,n_produtos,preco_medio,preco_mediana,preco_min,preco_max
quadrante,,,,,
↗ Alta Recompra + Alto LTV,35,244.08,251.07,74.57,427.03
↖ Baixa Recompra + Alto LTV,11,151.29,154.51,30.78,365.14
↘ Alta Recompra + Baixo LTV,10,164.21,133.74,80.55,314.63
↙ Baixa Recompra + Baixo LTV,36,105.45,113.31,23.68,210.76


In [26]:
# ── 4.3c Violin: Distribuição de Preços por Quadrante (Recompra × LTV) ──

if "metric_taxa_recompra" in df_scorecard.columns and "ltv_medio_ate_primeira_compra" in df_scorecard.columns:
    df_quad_price = (
        df_scorecard
        .loc[df_scorecard["product_name"] != "Insider Wash Bag"]
        .dropna(subset=["metric_taxa_recompra", "ltv_medio_ate_primeira_compra", "preco_medio_venda"])
        .copy()
    )

    # Referências: mesmas usadas no scatter 4.3b
    media_recompra = df_scorecard["metric_taxa_recompra"].dropna().mean()
    mediana_ltv = df_scorecard["ltv_medio_ate_primeira_compra"].dropna().median()

    # Classificar em quadrantes
    def _quadrante(row):
        alto_recompra = row["metric_taxa_recompra"] >= media_recompra
        alto_ltv = row["ltv_medio_ate_primeira_compra"] >= mediana_ltv
        if alto_recompra and alto_ltv:
            return "↗ Alta Recompra + Alto LTV"
        elif alto_recompra and not alto_ltv:
            return "↘ Alta Recompra + Baixo LTV"
        elif not alto_recompra and alto_ltv:
            return "↖ Baixa Recompra + Alto LTV"
        else:
            return "↙ Baixa Recompra + Baixo LTV"

    df_quad_price["quadrante"] = df_quad_price.apply(_quadrante, axis=1)

    # Ordem dos quadrantes para exibição
    quad_order = [
        "↗ Alta Recompra + Alto LTV",
        "↖ Baixa Recompra + Alto LTV",
        "↘ Alta Recompra + Baixo LTV",
        "↙ Baixa Recompra + Baixo LTV",
    ]

    fig_43c = px.violin(
        df_quad_price,
        x="quadrante",
        y="preco_medio_venda",
        color="quadrante",
        box=True,
        points="all",
        hover_data={"product_name": True, "category_4": True},
        category_orders={"quadrante": quad_order},
        color_discrete_map={
            "↗ Alta Recompra + Alto LTV": "#2ecc71",
            "↖ Baixa Recompra + Alto LTV": "#f39c12",
            "↘ Alta Recompra + Baixo LTV": "#3498db",
            "↙ Baixa Recompra + Baixo LTV": "#e74c3c",
        },
        title="Distribuição de Preço Médio por Quadrante (Recompra × LTV)<br>"
              f"<span style='font-size:10px;color:gray'>Cortes: Recompra = {media_recompra:.1%} (média) | LTV = R$ {mediana_ltv:.0f} (mediana)</span>",
        labels={"preco_medio_venda": "Preço Médio de Venda (R$)", "quadrante": ""},
    )
    fig_43c.update_layout(
        height=550,
        showlegend=False,
        yaxis_tickprefix="R$ ",
        yaxis_tickformat=",.0f",
    )

    # Contagem por quadrante no subtítulo
    for i, q in enumerate(quad_order):
        n = (df_quad_price["quadrante"] == q).sum()
        fig_43c.add_annotation(
            x=i, y=df_quad_price.loc[df_quad_price["quadrante"] == q, "preco_medio_venda"].max() * 1.05,
            text=f"n={n}", showarrow=False, font=dict(size=10, color="gray"),
        )

    fig_43c.show()

    # Resumo tabular
    resumo_quad = (
        df_quad_price.groupby("quadrante")["preco_medio_venda"]
        .agg(["count", "mean", "median", "min", "max"])
        .reindex(quad_order)
        .round(2)
    )
    resumo_quad.columns = ["n_produtos", "preco_medio", "preco_mediana", "preco_min", "preco_max"]
    display(resumo_quad)

,n_produtos,preco_medio,preco_mediana,preco_min,preco_max
quadrante,,,,,
↗ Alta Recompra + Alto LTV,35,244.08,251.07,74.57,427.03
↖ Baixa Recompra + Alto LTV,11,151.29,154.51,30.78,365.14
↘ Alta Recompra + Baixo LTV,10,164.21,133.74,80.55,314.63
↙ Baixa Recompra + Baixo LTV,36,105.45,113.31,23.68,210.76


In [27]:
# Boxplot: metric_ltv_ratio
if "metric_ltv_ratio" in df_scorecard.columns:
    fig_box_ltv = px.box(
        df_scorecard,
        y="metric_ltv_ratio",
        color="classificacao",
        color_discrete_map=COLOR_MAP_CLASSIFICACAO,
        title="Boxplot do LTV Ratio por Classificação",
        labels={"metric_ltv_ratio": "LTV Ratio", "classificacao": "Classificação"},
    )
    fig_box_ltv.show()

In [28]:
# ── 4.4 Evolução Share Trend — top 20 por receita ─────────────────

df_top20 = df_scorecard.nlargest(20, "receita_media_mensal").copy()

if "metric_share_trend" in df_top20.columns:
    df_top20 = df_top20.dropna(subset=["metric_share_trend"])
    df_top20 = df_top20.sort_values("metric_share_trend", ascending=True)
    df_top20["cor_tendencia"] = df_top20["metric_share_trend"].apply(
        lambda x: "#e74c3c" if x < -0.02 else "#2ecc71" if x > 0.02 else "#95a5a6"
    )

    fig_44 = px.bar(
        df_top20,
        x="metric_share_trend",
        y="product_name",
        orientation="h",
        color="cor_tendencia",
        color_discrete_map="identity",
        title="Tendência Share no Portfólio — Top 20 Produtos por Receita",
        labels={"metric_share_trend": "Share Trend (%)", "product_name": ""},
    )
    fig_44.add_vline(x=-0.02, line_dash="dash", line_color="red", opacity=0.5,
                     annotation_text="-2%")
    fig_44.add_vline(x=0.02, line_dash="dash", line_color="green", opacity=0.5,
                     annotation_text="+2%")
    fig_44.update_layout(showlegend=False)
    fig_44.show()

In [29]:
df_scorecard["metric_t_e_d_vs_categoria"]

0     0.897223
1     0.851465
2     0.927322
3     1.000000
4     1.153017
        ...   
88    1.181360
89    1.296823
90    0.862612
91    1.000000
92    1.590178
Name: metric_t_e_d_vs_categoria, Length: 93, dtype: float64

In [30]:
# ── 4.5 Devolução Relativa à Categoria ─────────────────────────────

if "metric_t_e_d_vs_categoria" in df_scorecard.columns:
    df_t_e_d = df_scorecard.dropna(subset=["metric_t_e_d_vs_categoria"]).copy()
    df_t_e_d = df_t_e_d.sort_values("metric_t_e_d_vs_categoria", ascending=False).head(30)

    fig_45 = px.bar(
        df_t_e_d,
        x="metric_t_e_d_vs_categoria",
        y="product_name",
        orientation="h",
        color="classificacao",
        color_discrete_map=COLOR_MAP_CLASSIFICACAO,
        title="T&D Relativa à Categoria (top 30 piores)",
        labels={"metric_t_e_d_vs_categoria": "T&D vs Categoria (x)", "product_name": ""},
    )
    fig_45.add_vline(x=1.0, line_dash="dash", line_color="gray", opacity=0.7,
                     annotation_text="1.0x (média)")
    fig_45.add_vline(x=1.5, line_dash="dash", line_color="red", opacity=0.7,
                     annotation_text="1.5x (alerta)")
    fig_45.show()

In [31]:
# ── 4.6 Risco de Estoque: Score × Cobertura ───────────────────────

if "metric_cobertura_dias" in df_scorecard.columns:
    df_risk = df_scorecard.dropna(subset=["score_total", "metric_cobertura_dias"]).copy()
    # Cap cobertura para visualização
    df_risk["cobertura_cap"] = df_risk["metric_cobertura_dias"].clip(upper=500)

    df_risk['size__receita_media_mensal'] = df_risk['receita_media_mensal']**0.5

    fig_46 = px.scatter(
        df_risk,
        x="score_total",
        y="cobertura_cap",
        size="size__receita_media_mensal",
        color="classificacao",
        color_discrete_map=COLOR_MAP_CLASSIFICACAO,
        hover_name="product_name",
        title="Risco de Estoque: Score Total × Cobertura (dias)",
        labels={"score_total": "Score Total", "cobertura_cap": "Cobertura (dias, cap 500)"},
    )
    # Zona de risco: score < 50 AND cobertura > 120d
    fig_46.add_hline(y=120, line_dash="dash", line_color="orange", opacity=0.5,
                     annotation_text="120d")
    fig_46.add_vline(x=50, line_dash="dash", line_color="orange", opacity=0.5,
                     annotation_text="Score 50")
    fig_46.add_shape(
        type="rect", x0=0, y0=120, x1=50, y1=500,
        fillcolor="red", opacity=0.05, line_width=0,
    )
    fig_46.update_layout(xaxis_range=[0, 100])
    fig_46.show()

In [32]:
# ── 4.7 Taxa de Recompra por Produto ──────────────────────────────

if "metric_taxa_recompra" in df_scorecard.columns:
    df_recomp = df_scorecard.dropna(subset=["metric_taxa_recompra"]).copy()
    df_recomp = df_recomp.sort_values("metric_taxa_recompra", ascending=True).tail(30)

    th_recomp = params["metrics"]["satisfacao"]["taxa_recompra"]["thresholds"]

    fig_47 = px.bar(
        df_recomp,
        x="metric_taxa_recompra",
        y="product_name",
        orientation="h",
        color="classificacao",
        color_discrete_map=COLOR_MAP_CLASSIFICACAO,
        title="Taxa de Recompra — Top 30 Produtos",
        labels={"metric_taxa_recompra": "Taxa de Recompra", "product_name": ""},
    )
    fig_47.add_vline(x=th_recomp["low"], line_dash="dash", line_color="orange",
                     opacity=0.7, annotation_text=f"{th_recomp['low']:.0%}")
    fig_47.add_vline(x=th_recomp["high"], line_dash="dash", line_color="green",
                     opacity=0.7, annotation_text=f"{th_recomp['high']:.0%}")
    fig_47.show()

In [33]:
# ── 4.8 LTV Ratio por Produto ─────────────────────────────────────

if "metric_ltv_ratio" in df_scorecard.columns:
    df_ltv = df_scorecard.dropna(subset=["metric_ltv_ratio"]).copy()
    df_ltv = df_ltv.sort_values("metric_ltv_ratio", ascending=True).tail(30)
    df_ltv["cor_ltv"] = df_ltv["metric_ltv_ratio"].apply(
        lambda x: "#2ecc71" if x >= 1.2 else "#3498db" if x >= 1.0 else "#95a5a6"
    )

    fig_48 = px.bar(
        df_ltv,
        x="metric_ltv_ratio",
        y="product_name",
        orientation="h",
        color="cor_ltv",
        color_discrete_map="identity",
        title="LTV Ratio por Produto — Top 30",
        labels={"metric_ltv_ratio": "LTV Ratio (vs média da base)", "product_name": ""},
    )
    fig_48.add_vline(x=1.0, line_dash="dash", line_color="gray", opacity=0.7,
                     annotation_text="1.0x (média)")
    fig_48.add_vline(x=1.2, line_dash="dash", line_color="green", opacity=0.5,
                     annotation_text="1.2x")
    fig_48.update_layout(showlegend=False)
    fig_48.show()

In [34]:
# ── 4.9 Ranking Final por Score Total ─────────────────────────────

df_ranking = df_scorecard.dropna(subset=["score_total"]).sort_values("score_total", ascending=True).copy()

fig_49 = px.bar(
    df_ranking,
    x="score_total",
    y="product_name",
    orientation="h",
    color="classificacao",
    color_discrete_map=COLOR_MAP_CLASSIFICACAO,
    title="Ranking Final — Score Total por Produto",
    labels={"score_total": "Score Total (0–100)", "product_name": ""},
)
fig_49.update_layout(height=max(400, len(df_ranking) * 22))
fig_49.show()

In [35]:
# ── 4.10 Breakdown por Pilar (heatmap) ────────────────────────────

pillar_cols = [c for c in df_scorecard.columns if c.startswith("score_pilar__")]
df_heat = df_scorecard.dropna(subset=["score_total"]).copy()
df_heat = df_heat.sort_values("score_total", ascending=False)

heat_data = df_heat.set_index("product_name")[pillar_cols]
heat_data.columns = [c.replace("score_pilar__", "").replace("_", " ").title() for c in heat_data.columns]

fig_410 = go.Figure(data=go.Heatmap(
    z=heat_data.values,
    x=heat_data.columns.tolist(),
    y=heat_data.index.tolist(),
    colorscale="RdYlGn",
    zmin=0,
    zmax=100,
    text=heat_data.round(0).astype(str).values,
    texttemplate="%{text}",
    hovertemplate="Produto: %{y}<br>Pilar: %{x}<br>Score: %{z:.0f}<extra></extra>",
))
fig_410.update_layout(
    title="Breakdown por Pilar — Score de cada produto",
    height=max(400, len(heat_data) * 22),
    yaxis=dict(autorange="reversed"),
)
fig_410.show()

In [36]:
# ── 4.11 Produtos por Área Responsável ────────────────────────────

if "responsavel_acionavel" in df_scorecard.columns:
    df_resp = df_scorecard.dropna(subset=["responsavel_acionavel"]).copy()

    fig_411 = px.histogram(
        df_resp,
        x="responsavel_acionavel",
        color="classificacao",
        color_discrete_map=COLOR_MAP_CLASSIFICACAO,
        barmode="stack",
        title="Produtos com Acionável por Área Responsável",
        labels={"responsavel_acionavel": "Área Responsável", "count": "Qtd Produtos"},
    )
    fig_411.update_layout(xaxis_tickangle=-30)
    fig_411.show()

In [37]:
# ── 4.12 Produtos com Flags Especiais (Breakthrough, Acessório) ───

df_flags = df_scorecard[
    df_scorecard["is_breakthrough"] | df_scorecard["is_accessory"]
].copy()

if len(df_flags) > 0:
    df_flags["flag"] = np.where(
        df_flags["is_breakthrough"], "Breakthrough",
        np.where(df_flags["is_accessory"], "Acessório", "—")
    )
    display_cols = ["product_name", "classificacao", "score_total", "flag"]
    if "cluster" in df_flags.columns:
        display_cols.insert(1, "cluster")

    print("⚠ Produtos com contexto qualitativo adicional — scorecard não é critério único de decisão:")
    display(df_flags[display_cols].sort_values("score_total", ascending=False))
else:
    print("Nenhum produto com flag especial identificado.")

⚠ Produtos com contexto qualitativo adicional — scorecard não é critério único de decisão:


,product_name,cluster,classificacao,score_total,flag
84,HighTech T-shirt LifeProof Feminino,Breakthrough,INVEST,78.666667,Breakthrough
48,HighTech T-shirt LifeProof Masculino,Breakthrough,KEEP,74.000000,Breakthrough


## 5. Exportação para Google Sheets

Exporta o scorecard completo para a planilha de perenes.

In [38]:
# ── 5.x Catálogo de colunas para exportação no Sheets ─────────────
# Bloco único consumido pelas rotinas de export (long e main infos).
# Cada coluna pode definir:
# - group: grande grupo visual da linha 1
# - number_format: formatação do Sheets (Type + pattern)
# - order: prioridade para ordenação (menor = mais à esquerda)

COLUMN_EXPORT_CATALOG = {
    # ===== Identidade =====
    "product_name": {"group": "Identidade", "order": 10},
    "category_1": {"group": "Identidade", "order": 11},
    "category_2": {"group": "Identidade", "order": 12},
    "category_3": {"group": "Identidade", "order": 13},
    "category_4": {"group": "Identidade", "order": 14},
    "cluster": {"group": "Identidade", "order": 15},
    "is_breakthrough": {"group": "Identidade", "order": 16},
    "is_accessory": {"group": "Identidade", "order": 17},
    "flag_mc3_atribuicao_fragil": {"group": "Identidade", "order": 18},

    # ===== Resultado =====
    "classificacao": {"group": "Resultado", "order": 20},
    "score_total": {
        "group": "Resultado",
        "order": 21,
        "number_format": {"type": "NUMBER", "pattern": "0.0"},
    },
    "fvet_override": {"group": "Resultado", "order": 22},
    "fvet_reason": {"group": "Resultado", "order": 23},
    "data_calculo": {"group": "Resultado", "order": 24},

    # ===== Score por Pilar =====
    "score_pilar__unit_economics": {
        "group": "Score por Pilar",
        "order": 30,
        "number_format": {"type": "NUMBER", "pattern": "0.0"},
    },
    "score_pilar__estoque": {
        "group": "Score por Pilar",
        "order": 31,
        "number_format": {"type": "NUMBER", "pattern": "0.0"},
    },
    "score_pilar__tracao_comercial": {
        "group": "Score por Pilar",
        "order": 32,
        "number_format": {"type": "NUMBER", "pattern": "0.0"},
    },
    "score_pilar__satisfacao_e_marca": {
        "group": "Score por Pilar",
        "order": 33,
        "number_format": {"type": "NUMBER", "pattern": "0.0"},
    },

    # ===== Routing =====
    "pilar_mais_baixo": {"group": "Routing", "order": 40},
    "responsavel_acionavel": {"group": "Routing", "order": 41},

    # ===== Unit Economics =====
    # "receita_total": {
    #     "group": "Unit Economics",
    #     "order": 50,
    #     "number_format": {"type": "CURRENCY", "pattern": "\"R$\" #,##0"},
    # },
    # "unidades_total": {
    #     "group": "Unit Economics",
    #     "order": 51,
    #     "number_format": {"type": "NUMBER", "pattern": "#,##0"},
    # },
    "receita_media_mensal": {
        "group": "Unit Economics",
        "order": 52,
        "number_format": {"type": "CURRENCY", "pattern": "\"R$\" #,##0"},
    },
    "metric_mc3": {
        "group": "Unit Economics",
        "order": 53,
        "number_format": {"type": "PERCENT", "pattern": "0.0%"},
    },
    "cmv_medio": {
        "group": "Unit Economics",
        "order": 54,
        "number_format": {"type": "CURRENCY", "pattern": "\"R$\" #,##0"},
    }, 
    "preco_cheio_medio": {
        "group": "Unit Economics",
        "order": 55,
        "number_format": {"type": "CURRENCY", "pattern": "\"R$\" #,##0"},
    },
    "markup_entrada": {
        "group": "Unit Economics",
        "order": 56,
        "number_format": {"type": "NUMBER", "pattern": "0.00"},
    }, 
    "markup_real": {
        "group": "Unit Economics",
        "order": 57,
        "number_format": {"type": "NUMBER", "pattern": "0.00"},
    }, 
    "metric_markup_vs_target": {
        "group": "Unit Economics",
        "order": 58,
        "number_format": {"type": "PERCENT", "pattern": "0.0%"},
    },
    "metric_receita_por_skc_ativo": {
        "group": "Unit Economics",
        "order": 59,
        "number_format": {"type": "NUMBER", "pattern": "0.00"},
    },
    "metric_contribuicao_absoluta": {
        "group": "Unit Economics",
        "order": 60,
        "number_format": {"type": "CURRENCY", "pattern": "\"R$\" #,##0"},
    },
    "metric_full_price_pct": {
        "group": "Unit Economics",
        "order": 61,
        "number_format": {"type": "PERCENT", "pattern": "0.0%"},
    },

    # ===== Estoque =====
    "metric_cobertura_dias": {
        "group": "Estoque",
        "order": 65,
        "number_format": {"type": "NUMBER", "pattern": "#,##0"},
    },
    "metric_disponibilidade_sku_semana": {
        "group": "Estoque",
        "order": 66,
        "number_format": {"type": "PERCENT", "pattern": "0.0%"},
    },
    "metric_cv_vendas_skc": {
        "group": "Estoque",
        "order": 67,
        "number_format": {"type": "NUMBER", "pattern": "0.00"},
    },
    "metric_ruptura_pct": {
        "group": "Estoque",
        "order": 68,
        "number_format": {"type": "PERCENT", "pattern": "0.0%"},
    },

    # ===== Tracao Comercial =====
    "metric_share_trend": {
        "group": "Tracao Comercial",
        "order": 70,
        "number_format": {"type": "NUMBER", "pattern": "0"},
    },
    "preco_medio_venda": {
        "group": "Tracao Comercial",
        "order": 71,
        "number_format": {"type": "CURRENCY", "pattern": "\"R$\" #,##0"},
    },

    # ===== Satisfacao & Marca =====
    "metric_taxa_recompra": {
        "group": "Satisfacao & Marca",
        "order": 80,
        "number_format": {"type": "PERCENT", "pattern": "0.0%"},
    },
    "metric_ltv_ratio": {
        "group": "Satisfacao & Marca",
        "order": 81,
        "number_format": {"type": "NUMBER", "pattern": "0.00"},
    },
    "t_e_d_produto": {
        "group": "Satisfacao & Marca",
        "order": 83,
        "number_format": {"type": "PERCENT", "pattern": "0.0%"},
    },

    # ===== Health =====
    "health__contribuicao_absoluta_raw": {
        "group": "Health Metrics",
        "order": 90,
        "number_format": {"type": "CURRENCY", "pattern": "\"R$\" #,##0"},
    },
}

GROUP_COLORS = {
    "Identidade":         {"red": 0.85, "green": 0.85, "blue": 0.85},
    "Resultado":          {"red": 0.73, "green": 0.84, "blue": 0.95},
    "Score por Pilar":    {"red": 0.71, "green": 0.88, "blue": 0.80},
    "Routing":            {"red": 0.95, "green": 0.87, "blue": 0.73},
    "Unit Economics":     {"red": 0.91, "green": 0.79, "blue": 0.97},
    "Estoque":            {"red": 0.79, "green": 0.90, "blue": 0.97},
    "Tracao Comercial":   {"red": 0.84, "green": 0.97, "blue": 0.84},
    "Satisfacao & Marca": {"red": 0.97, "green": 0.84, "blue": 0.79},
    "Health Metrics":     {"red": 1.00, "green": 0.95, "blue": 0.80},
    "Outros":             {"red": 0.93, "green": 0.93, "blue": 0.93},
}

def get_column_group(col: str) -> str:
    if col in COLUMN_EXPORT_CATALOG:
        return COLUMN_EXPORT_CATALOG[col]["group"]
    if col.startswith("score__"):
        return "Score por Pilar"
    if col.startswith("valor__") or col.startswith("faixa__"):
        # Valor/faixa herda grupo pelo slug do pilar no nome da coluna
        if "__unit_economics__" in col:
            return "Unit Economics"
        if "__estoque__" in col:
            return "Estoque"
        if "__tracao_comercial__" in col:
            return "Tracao Comercial"
        if "__satisfacao_e_marca__" in col:
            return "Satisfacao & Marca"
    if col.startswith("health__"):
        return "Health Metrics"
    return "Outros"

def get_column_order(col: str) -> int:
    if col in COLUMN_EXPORT_CATALOG:
        return COLUMN_EXPORT_CATALOG[col].get("order", 999)
    # Regras de fallback para colunas não mapeadas explicitamente
    if col.startswith("score__"):
        return 35
    if col.startswith("valor__"):
        return 100
    if col.startswith("faixa__"):
        return 110
    return 999

def get_column_number_format(col: str):
    if col in COLUMN_EXPORT_CATALOG:
        return COLUMN_EXPORT_CATALOG[col].get("number_format")
    if col.startswith("score__"):
        return {"type": "NUMBER", "pattern": "0.0"}
    return None

def format_export_header(col: str) -> str:
    return str(col).replace("__", "\n").replace("_", " " ).lower()

def build_group_row_and_spans(columns: list[str]):
    groups = [get_column_group(c) for c in columns]
    row = [""] * len(columns)
    spans = []
    current = groups[0]
    start = 0
    for i, g in enumerate(groups[1:], start=1):
        if g != current:
            row[start] = current
            spans.append((current, start, i))
            current = g
            start = i
    row[start] = current
    spans.append((current, start, len(columns)))
    return row, spans

In [39]:

# ── Exportar Scorecard para Sheets ─────────────────────────────────
df_exp_long = df_scorecard.copy()
if SPREADSHEET_SCORECARD is None:
    print("⚠ Google Sheets não autenticado — exportação ignorada.")
else:
    ws_long = SPREADSHEET_SCORECARD.worksheet("import deepnote long")

    df_exp_long["data_calculo"] = date.today().isoformat()

    # Converter tipos incompatíveis com Sheets
    for col in df_exp_long.columns:
        if pd.api.types.is_extension_array_dtype(df_exp_long[col]):
            df_exp_long[col] = df_exp_long[col].astype(object)
        if pd.api.types.is_bool_dtype(df_exp_long[col]):
            df_exp_long[col] = df_exp_long[col].astype(str)

    # Ordenar linhas por classificação + score
    class_order = {"INVEST": 0, "KEEP": 1, "WATCH": 2, "AT RISK": 3}
    df_exp_long["_sort"] = df_exp_long["classificacao"].map(class_order)
    df_exp_long.sort_values(["_sort", "score_total"], ascending=[True, False], inplace=True)
    df_exp_long.drop(columns="_sort", inplace=True)

    df_exp_long = df_exp_long.fillna(" ")

    # Ordenar colunas por relevância/grupo via catálogo
    long_cols_sorted = sorted(
        df_exp_long.columns.tolist(),
        key=lambda c: (get_column_order(c), get_column_group(c), c),
    )
    # Garantir classificação na primeira posição
    if "classificacao" in long_cols_sorted:
        long_cols_sorted.remove("classificacao")
        long_cols_sorted = ["classificacao"] + long_cols_sorted

    df_exp_long = df_exp_long[long_cols_sorted]
    cols_long = [format_export_header(c) for c in df_exp_long.columns]

    # Linha 1 = grupos, linha 2 = headers
    group_row_long, group_spans_long = build_group_row_and_spans(df_exp_long.columns.tolist())

    ws_long.clear()
    ws_long.update(range_name="A1", values=[group_row_long] + [cols_long] + df_exp_long.values.tolist())

    # Formatação da aba long (cores de grupo + number formats + freeze)
    sheet_id_long = ws_long.id
    n_data_rows_long = len(df_exp_long)
    requests_long = []

    for group_label, start_ci, end_ci in group_spans_long:
        bg = GROUP_COLORS.get(group_label, GROUP_COLORS["Outros"])
        requests_long.append({
            "repeatCell": {
                "range": {
                    "sheetId": sheet_id_long,
                    "startRowIndex": 0, "endRowIndex": 1,
                    "startColumnIndex": start_ci, "endColumnIndex": end_ci,
                },
                "cell": {
                    "userEnteredFormat": {
                        "backgroundColor": bg,
                        "horizontalAlignment": "LEFT",
                        "verticalAlignment": "MIDDLE",
                        "textFormat": {"bold": True, "fontSize": 10},
                    }
                },
                "fields": "userEnteredFormat(backgroundColor,horizontalAlignment,verticalAlignment,textFormat)",
            }
        })

    for ci, col in enumerate(df_exp_long.columns):
        fmt = get_column_number_format(col)
        if fmt is None:
            continue
        requests_long.append({
            "repeatCell": {
                "range": {
                    "sheetId": sheet_id_long,
                    "startRowIndex": 2, "endRowIndex": 2 + n_data_rows_long,
                    "startColumnIndex": ci, "endColumnIndex": ci + 1,
                },
                "cell": {"userEnteredFormat": {"numberFormat": fmt}},
                "fields": "userEnteredFormat.numberFormat",
            }
        })

    requests_long.append({
        "updateSheetProperties": {
            "properties": {
                "sheetId": sheet_id_long,
                "gridProperties": {"frozenRowCount": 2},
            },
            "fields": "gridProperties.frozenRowCount",
        }
    })

    SPREADSHEET_SCORECARD.batch_update({"requests": requests_long})

    print(f"✓ Scorecard exportado ({len(df_exp_long)} produtos → aba 'import deepnote long')")
    print("  └─ Linha 1: grupos com cores | Linha 2: cabeçalhos | Dados: linha 3+")

    # ══════════════════════════════════════════════════════════════
    # Aba enxugada: deepnote main infos
    # ══════════════════════════════════════════════════════════════

    MAIN_COLS = [
        "product_name", "category_4", "cluster",
        "classificacao", "score_total",
        "score_pilar__unit_economics", "score_pilar__estoque",
        "score_pilar__tracao_comercial", "score_pilar__satisfacao_e_marca",
        "pilar_mais_baixo", "responsavel_acionavel",
        "metric_mc3", "metric_markup_vs_target",
        "metric_cobertura_dias", "metric_disponibilidade_sku_semana",
        "metric_taxa_recompra", "metric_t_e_d_vs_categoria", "t_e_d_produto", "metric_share_trend",
        "receita_media_mensal", "fvet_override", "fvet_reason", "data_calculo",
    ]

    main_cols_available = [c for c in MAIN_COLS if c in df_exp_long.columns]
    # Reordena também dentro da aba principal pelo catálogo (mantendo escopo das MAIN_COLS)
    main_cols_available = sorted(main_cols_available, key=lambda c: (get_column_order(c), get_column_group(c), c))
    if "classificacao" in main_cols_available:
        main_cols_available.remove("classificacao")
        main_cols_available = ["classificacao"] + main_cols_available

    df_exp_main = df_exp_long[main_cols_available].copy()
    cols_main = [format_export_header(c) for c in main_cols_available]
    group_row_main, group_spans_main = build_group_row_and_spans(main_cols_available)

    try:
        ws_main = SPREADSHEET_SCORECARD.worksheet("deepnote main infos")
    except Exception:
        ws_main = SPREADSHEET_SCORECARD.add_worksheet(
            title="deepnote main infos", rows=500, cols=len(main_cols_available) + 5
        )

    ws_main.clear()
    ws_main.update(range_name="A1", values=[group_row_main] + [cols_main] + df_exp_main.values.tolist())

    sheet_id_main = ws_main.id
    n_data_rows_main = len(df_exp_main)
    requests_main = []

    for group_label, start_ci, end_ci in group_spans_main:
        bg = GROUP_COLORS.get(group_label, GROUP_COLORS["Outros"])
        requests_main.append({
            "repeatCell": {
                "range": {
                    "sheetId": sheet_id_main,
                    "startRowIndex": 0, "endRowIndex": 1,
                    "startColumnIndex": start_ci, "endColumnIndex": end_ci,
                },
                "cell": {
                    "userEnteredFormat": {
                        "backgroundColor": bg,
                        "horizontalAlignment": "LEFT",
                        "verticalAlignment": "MIDDLE",
                        "textFormat": {"bold": True, "fontSize": 10},
                    }
                },
                "fields": "userEnteredFormat(backgroundColor,horizontalAlignment,verticalAlignment,textFormat)",
            }
        })

    for ci, col in enumerate(main_cols_available):
        fmt = get_column_number_format(col)
        if fmt is None:
            continue
        requests_main.append({
            "repeatCell": {
                "range": {
                    "sheetId": sheet_id_main,
                    "startRowIndex": 2, "endRowIndex": 2 + n_data_rows_main,
                    "startColumnIndex": ci, "endColumnIndex": ci + 1,
                },
                "cell": {"userEnteredFormat": {"numberFormat": fmt}},
                "fields": "userEnteredFormat.numberFormat",
            }
        })

    requests_main.append({
        "updateSheetProperties": {
            "properties": {
                "sheetId": sheet_id_main,
                "gridProperties": {"frozenRowCount": 2},
            },
            "fields": "gridProperties.frozenRowCount",
        }
    })

    SPREADSHEET_SCORECARD.batch_update({"requests": requests_main})

    print(f"✓ Main infos exportado ({len(df_exp_main)} produtos, {len(main_cols_available)} colunas → aba 'deepnote main infos')")
    print("  └─ Linha 1: grupos com cores | Linha 2: cabeçalhos | Dados: linha 3+")


✓ Scorecard exportado (93 produtos → aba 'import deepnote long')
  └─ Linha 1: grupos com cores | Linha 2: cabeçalhos | Dados: linha 3+
✓ Main infos exportado (93 produtos, 23 colunas → aba 'deepnote main infos')
  └─ Linha 1: grupos com cores | Linha 2: cabeçalhos | Dados: linha 3+


In [40]:
df_exp_long.to_csv('outputs/scorecard_perenes.csv', index=False)

df_exp_long.loc[df_exp_long['product_name'].str.contains('Cueca')][['product_name', 'metric_cobertura_dias']]

,product_name,metric_cobertura_dias
16,Cueca Boxer Comfort Simples Masculino,
18,Cueca Boxer Performance Simples Masculino,
21,Cueca Boxer Performance Anti Suor Masculino,
22,Cueca Boxer Comfort Anti Suor Masculino,
51,Cueca Slip Comfort Simples Masculino,


In [41]:
import pandas as pd

df_exp_long = df_exp_long.copy()

cols_not_numeric = {
    # Dimensões textuais
    "classificacao",
    "product_name",
    "category_1",
    "category_2",
    "category_3",
    "category_4",
    "cluster",

    # Flags / booleanos
    "is_breakthrough",
    "is_accessory",
    "flag_mc3_atribuicao_fragil",
    "flag_markup",

    # Campos textuais de decisão
    "fvet_reason",
    "pilar_mais_baixo",
    "responsavel_acionavel",

    # Classificações por faixa
    "faixa__estoque__cobertura_dias",
    "faixa__estoque__cv_vendas_skc",
    "faixa__estoque__disponibilidade_sku_semana",
    "faixa__satisfacao_e_marca__ltv_ratio",
    "faixa__satisfacao_e_marca__t_e_d_vs_categoria",
    "faixa__satisfacao_e_marca__taxa_recompra",
    "faixa__tracao_comercial__full_price_pct",
    "faixa__tracao_comercial__share_trend",
    "faixa__unit_economics__contribuicao_absoluta",
    "faixa__unit_economics__markup_vs_target",
    "faixa__unit_economics__mc3",

    # Outros campos aparentemente categóricos
    "category_4_mkp",
    "cobertura_fonte",

    # Datas
    "data_calculo",
    "ingestion_date",
}

cols_to_numeric = [
    col
    for col in df_exp_long.columns
    if col not in cols_not_numeric
]

df_exp_long[cols_to_numeric] = (
    df_exp_long[cols_to_numeric]
    .apply(pd.to_numeric, errors="coerce")
    .astype("float64")
)

# Datas
df_exp_long["data_calculo"] = pd.to_datetime(
    df_exp_long["data_calculo"],
    errors="coerce",
)

df_exp_long["ingestion_date"] = pd.Timestamp.today().strftime('%Y-%m-%d')


# Export para o BQ

In [42]:
from local_bigquery_utils import load_df_to_bigquery

load_df_to_bigquery(
    df=df_exp_long,
    project_id="insider-data-lake",
    dataset_id="sop_silver",
    table_id="scorecard_perenes_history",
    partition_field="ingestion_date",
    service_account_info=service_account_info,
)

Partition 20260817 successfully written to insider-data-lake.sop_silver.scorecard_perenes_history.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5022f744-2b88-47b4-b01f-4ff0a47ee9d0' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>